In [1]:
# ============================================================
# Imports and Settings
# ============================================================

from pathlib import Path
from datetime import datetime
import importlib

import osxphotos

# Reload helper module without restarting the Jupyter kernel.
#
# This is important because:
# - In Jupyter, `from explorephotoslibrary import *` does NOT automatically
#   pick up edits made to explorephotoslibrary.py after the first import.
# - Restarting the kernel would lose the current notebook state.
# - Reloading the module here lets later cells use the updated functions.
import explorephotoslibrary as _explorephotoslibrary
importlib.reload(_explorephotoslibrary)

from explorephotoslibrary import *


USE_INVENTORY_CACHE = True


# Rebuild only the libraries listed here.
#
# Normal use:
# FORCE_REBUILD_INVENTORY_KEYS = set()
#
# Rebuild Current Default only:
# FORCE_REBUILD_INVENTORY_KEYS = {"current_default"}
#
# Rebuild both:
# FORCE_REBUILD_INVENTORY_KEYS = {
#     "backup_20250317",
#     "current_default",
# }
FORCE_REBUILD_INVENTORY_KEYS = {
    "current_default",
    # "backup_20250317",
}


# Path-selection policy:
#
# False:
# - Reuse each library path saved in:
#     data/local_config/test2_library_paths.json
#
# True:
# - Reselect the path only for libraries listed in
#   FORCE_REBUILD_INVENTORY_KEYS.
# - Libraries not listed in FORCE_REBUILD_INVENTORY_KEYS continue
#   using their saved paths.
#
# Current situation:
# - Current Default Photos Library was moved.
# - Backup Photos Library did not move.
# Therefore:
#     FORCE_REBUILD_INVENTORY_KEYS = {"current_default"}
#     FORCE_RESELECT_LIBRARY_PATHS = True
FORCE_RESELECT_LIBRARY_PATHS = False


TEST2_LIBRARY_PROMPTS = {
    "backup_20250317": (
        "Select BACKUP Photos Library: backup_20250317"
    ),
    "current_default": (
        "Select CURRENT default Photos Library: current_default"
    ),
}


TEST2_DEFAULT_INITIAL_DIRS = {
    "backup_20250317": "/Volumes",
    "current_default": str(Path.home() / "Pictures"),
}


# Short visible confirmation that this settings cell really ran.
{
    "use_inventory_cache": USE_INVENTORY_CACHE,
    "force_rebuild_inventory_keys": sorted(
        FORCE_REBUILD_INVENTORY_KEYS
    ),
    "force_reselect_library_paths": (
        FORCE_RESELECT_LIBRARY_PATHS
    ),
    "libraries_that_will_be_reselected": (
        sorted(FORCE_REBUILD_INVENTORY_KEYS)
        if FORCE_RESELECT_LIBRARY_PATHS
        else []
    ),
}

{'use_inventory_cache': True,
 'force_rebuild_inventory_keys': ['current_default'],
 'force_reselect_library_paths': False,
 'libraries_that_will_be_reselected': []}

In [2]:
# =====================================================================================
# Load or build inventories — unique ID generation / validation included
# =====================================================================================

TEST2_LIBRARY_HISTORY_PATH = Path("data/local_config/test2_library_paths.json")

def get_test2_library_path(library_key):
    library_history = load_json_file(
        TEST2_LIBRARY_HISTORY_PATH,
        default={},
    ) or {}

    saved_library_path = library_history.get(library_key)

    # FORCE_RESELECT_LIBRARY_PATHS is one global True/False switch.
    #
    # When True, reselect only libraries also listed in
    # FORCE_REBUILD_INVENTORY_KEYS.
    should_force_reselect = (
        FORCE_RESELECT_LIBRARY_PATHS
        and library_key in FORCE_REBUILD_INVENTORY_KEYS
    )

    if (
        saved_library_path
        and Path(saved_library_path).exists()
        and not should_force_reselect
    ):
        library_path = Path(saved_library_path)

        print("=" * 80)
        print(f"Use saved Photos Library path for: {library_key}")
        print("=" * 80)
        print(f"{library_key} library path:", library_path)
        print()

        return library_path

    if saved_library_path:
        initial_dir = Path(saved_library_path).parent
    else:
        initial_dir = Path(
            TEST2_DEFAULT_INITIAL_DIRS.get(
                library_key,
                "/Volumes",
            )
        )

    prompt = TEST2_LIBRARY_PROMPTS.get(
        library_key,
        f"Select Photos Library for: {library_key}",
    )

    print("=" * 80)

    if should_force_reselect:
        print(f"Force reselect Photos Library path for: {library_key}")
    else:
        print(prompt)

    print("=" * 80)

    library_path = Path(
        choose_photos_library_path(
            initial_dir=initial_dir,
            prompt=prompt,
        )
    )

    library_history[library_key] = str(library_path)
    library_history[f"{library_key}_selected_at"] = (
        datetime.now().isoformat()
    )

    save_json_file(
        TEST2_LIBRARY_HISTORY_PATH,
        library_history,
    )

    print(f"{library_key} library path:", library_path)
    print()

    return library_path

def print_section2_identity_summary(inventory, label):
    key_to_count = {}
    assets_without_unique_id = 0

    for asset in inventory.get("assets") or []:
        unique_id = asset.get("photo_library_asset_unique_id")

        if unique_id is None:
            assets_without_unique_id += 1
            continue

        key_to_count[unique_id] = key_to_count.get(unique_id, 0) + 1

    duplicate_group_count = sum(
        1
        for count in key_to_count.values()
        if count > 1
    )

    duplicate_asset_count = sum(
        count
        for count in key_to_count.values()
        if count > 1
    )

    is_ok = (
        assets_without_unique_id == 0
        and duplicate_group_count == 0
    )

    print()
    print(f"{label} identity summary")
    print("-" * 80)
    print("total asset count:", len(inventory.get("assets") or []))
    print("generated unique ID count:", len(key_to_count))
    print("assets without unique ID:", assets_without_unique_id)
    print("duplicate unique ID group count:", duplicate_group_count)
    print("duplicate asset count:", duplicate_asset_count)
    print("photo_library_asset_unique_id status:", "OK" if is_ok else "FAILED")

    if not is_ok:
        raise RuntimeError(f"{label} inventory identity validation failed.")


def load_or_build_inventory(library_key):
    library_path = get_test2_library_path(library_key)
    should_rebuild_inventory = library_key in FORCE_REBUILD_INVENTORY_KEYS

    if USE_INVENTORY_CACHE and not should_rebuild_inventory:
        print("=" * 80)
        print(f"Load inventory cache: {library_key}")
        print("=" * 80)

        try:
            inventory = load_inventory_cache(library_key)
            print_section2_identity_summary(inventory, library_key)
            return inventory
        except FileNotFoundError:
            print(f"Cache not found for {library_key}. Build inventory instead.")
            print()

    if should_rebuild_inventory:
        print("=" * 80)
        print(f"Force rebuild inventory: {library_key}")
        print("=" * 80)
    else:
        print("=" * 80)
        print(f"Build inventory: {library_key}")
        print("=" * 80)

    osx_assets = osxphotos.PhotosDB(str(library_path)).photos()
    print(f"{library_key} osx asset count:", len(osx_assets))

    inventory = build_inventory(osx_assets)

    print()
    print(f"{library_key} inventory summary")
    print("-" * 80)
    print_inventory_summary(inventory)
    print_section2_identity_summary(inventory, library_key)

    save_inventory_cache(inventory, library_key)

    return inventory


inventory_backup = load_or_build_inventory("backup_20250317")

print()

inventory_current = load_or_build_inventory("current_default")

Use saved Photos Library path for: backup_20250317
backup_20250317 library path: /Volumes/PRO-G40-0605/Photos Library-iCloud-20250317（iCloud 20250325崩潰前最後的備份）--opened by macOS Sequoia on 20260604/Photos Library-iCloud-20250317（iCloud 20250325崩潰前最後的備份）--opened by macOS Sequoia on 20260604.photoslibrary

Load inventory cache: backup_20250317
loaded inventory cache: data/inventory_cache/backup_20250317.inventory.pkl.gz
elapsed seconds: 0.67
inventory assets: 71572
inventory albums: 5171
inventory folders: 35
special assets:
  PATH_MISSING: 0
  PATH_NONE: 0
  SYNDICATED_NO_NORMAL_ORIGINAL: 0
  UNKNOWN_PATH: 0
movies: 6224
hidden: 0
favorites: 701
descriptions: 728
keywords: 23747

backup_20250317 identity summary
--------------------------------------------------------------------------------
total asset count: 71572
generated unique ID count: 71572
assets without unique ID: 0
duplicate unique ID group count: 0
duplicate asset count: 0
photo_library_asset_unique_id status: OK

Use saved Ph

In [ ]:
# ============================================================
# Whole-library folder/album membership comparison report
# ============================================================

folder_album_membership_report = write_folder_album_membership_comparison_report(
    inventory_backup=inventory_backup,
    inventory_current=inventory_current,
    report_root=Path("reports/test2_folder_album_membership_comparison"),
    label="whole_library",
    target_root_folder_path=None,
    include_ok_rows=False,
)

{
    "folder_album_membership_report": folder_album_membership_report,
}

In [5]:
# ============================================================
# Current Default — Duplicate Album Titles
# Numbers-friendly TSV report
#
# Duplicate title definition:
# - exact same title, OR
# - title differs only by whitespace formatting:
#   leading/trailing spaces, repeated spaces, tabs, line breaks,
#   or other Unicode whitespace
# - 2 or more different Current album UUIDs
#
# IMPORTANT:
# - punctuation is NOT ignored
# - "..." and "…" remain meaningful
# - emoji and hashtags remain meaningful
#
# READ ONLY:
# - reads inventory_current only
# - does not rebuild inventory
# - does not modify Photos Library
# ============================================================

from collections import defaultdict, Counter
from pathlib import Path
from datetime import datetime
import csv
import hashlib


def duplicate_album_safe_tsv_text(value):
    """
    Preserve the complete original text while keeping each album
    on one TSV row.

    Actual tabs and line breaks inside a title are represented
    visibly as \\t, \\r, and \\n.
    """
    return (
        str(value or "")
        .replace("\\", "\\\\")
        .replace("\t", "\\t")
        .replace("\r", "\\r")
        .replace("\n", "\\n")
    )


def duplicate_album_normalize_title_for_matching(value):
    """
    Normalize whitespace only.

    Examples treated as equivalent:

        "ABC   DEF"
        "ABC DEF"
        "ABC\\nDEF"
        " ABC DEF   "

    This does NOT remove punctuation, dots, ellipsis,
    hashtags, emoji, or other visible characters.
    """
    text = str(value or "")

    return " ".join(text.split())


def duplicate_album_title_whitespace_stats(value):
    """
    Return useful whitespace diagnostics for one raw title.
    """
    text = str(value or "")

    leading_count = (
        len(text) - len(text.lstrip())
    )

    trailing_count = (
        len(text) - len(text.rstrip())
    )

    return {
        "raw_title_length":
            len(text),

        "normalized_title_length":
            len(
                duplicate_album_normalize_title_for_matching(
                    text
                )
            ),

        "leading_whitespace_count":
            leading_count,

        "trailing_whitespace_count":
            trailing_count,

        "tab_count":
            text.count("\t"),

        "line_feed_count":
            text.count("\n"),

        "carriage_return_count":
            text.count("\r"),

        "nonbreaking_space_count":
            text.count("\u00a0"),
    }


def duplicate_album_normalize_folder_path(path):
    text = str(path or "").strip()

    text = text.replace("\\", "/")
    text = text.replace(" / ", "/")

    return " / ".join(
        part.strip()
        for part in text.split("/")
        if part.strip()
    )


def duplicate_album_leaf_folder_paths(album):
    """
    Return only the deepest folder path or paths.

    Albums without a folder are reported as [ROOT_ALBUMS].
    """
    all_paths = {
        duplicate_album_normalize_folder_path(
            folder.get("path")
            or folder.get("title")
        )
        for folder in (
            album.get("folders") or {}
        ).values()
        if (
            folder.get("path")
            or folder.get("title")
        )
    }

    all_paths.discard("")

    if not all_paths:
        return ("[ROOT_ALBUMS]",)

    leaf_paths = [
        path
        for path in all_paths
        if not any(
            other != path
            and other.startswith(path + " / ")
            for other in all_paths
        )
    ]

    return tuple(sorted(leaf_paths))


def duplicate_album_membership_checksum(asset_uuids):
    payload = "\n".join(
        sorted(asset_uuids)
    )

    return hashlib.sha256(
        payload.encode("utf-8")
    ).hexdigest()[:16]


# ------------------------------------------------------------
# Build exact asset membership for every Current album UUID
# ------------------------------------------------------------

members_by_album_uuid = defaultdict(set)

for asset in inventory_current.get("assets") or []:
    asset_uuid = str(
        asset.get("uuid") or ""
    )

    if not asset_uuid:
        continue

    for album_uuid in (
        asset.get("albums") or {}
    ):
        members_by_album_uuid[
            str(album_uuid)
        ].add(asset_uuid)


# ------------------------------------------------------------
# Group albums by whitespace-normalized complete title
# ------------------------------------------------------------

albums_by_normalized_title = defaultdict(list)

for inventory_key, album in (
    inventory_current.get("albums") or {}
).items():
    raw_title = album.get("title")

    if raw_title is None:
        continue

    raw_title = str(raw_title)

    normalized_title = (
        duplicate_album_normalize_title_for_matching(
            raw_title
        )
    )

    if normalized_title == "":
        continue

    album_uuid = str(
        album.get("uuid")
        or inventory_key
    )

    member_set = set(
        members_by_album_uuid.get(
            album_uuid,
            set(),
        )
    )

    whitespace_stats = (
        duplicate_album_title_whitespace_stats(
            raw_title
        )
    )

    albums_by_normalized_title[
        normalized_title
    ].append({
        "album_uuid":
            album_uuid,

        "folder_paths":
            duplicate_album_leaf_folder_paths(
                album
            ),

        "raw_title":
            raw_title,

        "normalized_title":
            normalized_title,

        "member_set":
            member_set,

        "asset_count":
            len(member_set),

        "membership_checksum":
            duplicate_album_membership_checksum(
                member_set
            ),

        **whitespace_stats,
    })


duplicate_groups = [
    (
        normalized_title,
        album_objects,
    )
    for normalized_title, album_objects
    in albums_by_normalized_title.items()
    if len({
        row["album_uuid"]
        for row in album_objects
    }) >= 2
]

duplicate_groups.sort(
    key=lambda item: item[0]
)


# ------------------------------------------------------------
# Create one Numbers-friendly row per album object
# ------------------------------------------------------------

report_rows = []

exact_title_group_count = 0
whitespace_only_title_group_count = 0

exact_membership_group_count = 0
different_membership_group_count = 0


for group_number, (
    normalized_title,
    album_objects,
) in enumerate(
    duplicate_groups,
    start=1,
):
    album_objects = sorted(
        album_objects,
        key=lambda row: (
            row["folder_paths"],
            row["album_uuid"],
        ),
    )

    total_album_objects = len(
        album_objects
    )

    extra_copy_count = (
        total_album_objects - 1
    )

    raw_title_values = {
        row["raw_title"]
        for row in album_objects
    }

    if len(raw_title_values) == 1:
        title_match_type = (
            "EXACT_TITLE"
        )

        exact_title_group_count += 1

    else:
        title_match_type = (
            "WHITESPACE_ONLY_DIFFERENCE"
        )

        whitespace_only_title_group_count += 1


    membership_sets = [
        frozenset(row["member_set"])
        for row in album_objects
    ]

    membership_set_counts = Counter(
        membership_sets
    )

    distinct_membership_sets = set(
        membership_sets
    )

    if len(distinct_membership_sets) == 1:
        group_membership_status = (
            "EXACT_SAME_MEMBERSHIP"
        )

        exact_membership_group_count += 1

    else:
        group_membership_status = (
            "DIFFERENT_MEMBERSHIP"
        )

        different_membership_group_count += 1


    for copy_number, row in enumerate(
        album_objects,
        start=1,
    ):
        membership_key = frozenset(
            row["member_set"]
        )

        report_rows.append({
            "group_number":
                group_number,

            "copy_number_in_group":
                copy_number,

            "total_album_objects_in_group":
                total_album_objects,

            "extra_copy_count_in_group":
                extra_copy_count,

            "title_match_type":
                title_match_type,

            "group_membership_status":
                group_membership_status,

            "folder_path":
                " || ".join(
                    row["folder_paths"]
                ),

            "album_title":
                row["raw_title"],

            "raw_title_length":
                row["raw_title_length"],

            "normalized_title_length":
                row["normalized_title_length"],

            "leading_whitespace_count":
                row[
                    "leading_whitespace_count"
                ],

            "trailing_whitespace_count":
                row[
                    "trailing_whitespace_count"
                ],

            "tab_count":
                row["tab_count"],

            "line_feed_count":
                row["line_feed_count"],

            "carriage_return_count":
                row[
                    "carriage_return_count"
                ],

            "nonbreaking_space_count":
                row[
                    "nonbreaking_space_count"
                ],

            "album_uuid":
                row["album_uuid"],

            "asset_count":
                row["asset_count"],

            "membership_checksum":
                row[
                    "membership_checksum"
                ],

            "copies_with_same_membership":
                membership_set_counts[
                    membership_key
                ],
        })


# ------------------------------------------------------------
# Save Numbers-friendly TSV
# ------------------------------------------------------------

run_time = datetime.now()

report_dir = (
    Path(
        "reports/"
        "test2_current_duplicate_album_titles"
    )
    / f"{run_time:%Y%m%d-%H%M%S}"
)

report_dir.mkdir(
    parents=True,
    exist_ok=False,
)

tsv_path = (
    report_dir
    / "current_duplicate_album_titles.tsv"
).resolve()


fieldnames = [
    "group_number",
    "copy_number_in_group",
    "total_album_objects_in_group",
    "extra_copy_count_in_group",

    "title_match_type",
    "group_membership_status",

    "folder_path",
    "album_title",

    "raw_title_length",
    "normalized_title_length",

    "leading_whitespace_count",
    "trailing_whitespace_count",

    "tab_count",
    "line_feed_count",
    "carriage_return_count",
    "nonbreaking_space_count",

    "album_uuid",
    "asset_count",
    "membership_checksum",
    "copies_with_same_membership",
]


with tsv_path.open(
    "w",
    encoding="utf-8",
    newline="",
) as file:
    writer = csv.DictWriter(
        file,
        fieldnames=fieldnames,
        delimiter="\t",
        lineterminator="\n",
        quoting=csv.QUOTE_MINIMAL,
    )

    writer.writeheader()

    for row in report_rows:
        safe_row = dict(row)

        safe_row["folder_path"] = (
            duplicate_album_safe_tsv_text(
                safe_row["folder_path"]
            )
        )

        safe_row["album_title"] = (
            duplicate_album_safe_tsv_text(
                safe_row["album_title"]
            )
        )

        writer.writerow(safe_row)


# ------------------------------------------------------------
# Summary
# ------------------------------------------------------------

duplicate_album_title_result = {
    "tsv_path":
        str(tsv_path),

    "duplicate_title_group_count":
        len(duplicate_groups),

    "exact_title_group_count":
        exact_title_group_count,

    "whitespace_only_title_group_count":
        whitespace_only_title_group_count,

    "exact_same_membership_group_count":
        exact_membership_group_count,

    "different_membership_group_count":
        different_membership_group_count,

    "album_object_count_in_groups":
        len(report_rows),

    "total_extra_album_objects":
        sum(
            len(album_objects) - 1
            for _, album_objects
            in duplicate_groups
        ),
}


print(
    "Duplicate title groups:",
    duplicate_album_title_result[
        "duplicate_title_group_count"
    ],
)

print(
    "Exact-title groups:",
    duplicate_album_title_result[
        "exact_title_group_count"
    ],
)

print(
    "Whitespace-only title-difference groups:",
    duplicate_album_title_result[
        "whitespace_only_title_group_count"
    ],
)

print(
    "Exact same-membership groups:",
    duplicate_album_title_result[
        "exact_same_membership_group_count"
    ],
)

print(
    "Different-membership groups:",
    duplicate_album_title_result[
        "different_membership_group_count"
    ],
)

print(
    "Total extra album objects:",
    duplicate_album_title_result[
        "total_extra_album_objects"
    ],
)

print()
print("TSV report:")
print(tsv_path)

Duplicate title groups: 33
Exact-title groups: 30
Whitespace-only title-difference groups: 3
Exact same-membership groups: 9
Different-membership groups: 24
Total extra album objects: 104

TSV report:
/Users/huohsien/workspace/python/explore_photos_library/reports/test2_current_duplicate_album_titles/20260617-155503/current_duplicate_album_titles.tsv


In [6]:
# ============================================================
# Delete duplicate album shells — export strict PhotoKit manifest
#
# SAFE candidate definition:
# 1. Current has 2+ album objects whose titles are equal after
#    whitespace-only normalization.
# 2. Their complete Current asset memberships are exactly equal.
# 3. Backup has exactly one album with the same normalized title
#    and the same cross-library asset membership.
# 4. Exactly one Current album is at that Backup folder path.
# 5. The keeper's raw title exactly equals the Backup raw title.
# 6. Every delete candidate has exactly one different folder path.
#
# Excluded:
# - empty albums
# - albums with missing asset unique IDs
# - "#給資料夾置頂用"
#
# READ ONLY: writes TSV files only; does not modify Photos.
# ============================================================

from collections import defaultdict
from pathlib import Path
from datetime import datetime
import csv
import hashlib
import json
import shutil

DELETE_DUP_INBOX = Path.home() / "Downloads" / "PhotosRepairMVP_Inbox"
DELETE_DUP_MANIFEST = DELETE_DUP_INBOX / "DeleteDuplicateAlbumsManifestLatest.tsv"
DELETE_DUP_REVIEW = DELETE_DUP_INBOX / "DeleteDuplicateAlbumsCandidatesLatest.tsv"
DELETE_DUP_ARCHIVE = DELETE_DUP_INBOX / "archive"
DELETE_DUP_EXCLUDED_TITLES = {"#給資料夾置頂用"}
# 給定刪除範圍的Root Folder
DELETE_DUP_TARGET_KEEP_FOLDER_PATH = "NSFW"

WRITE_DELETE_DUP_ARCHIVE = True


def delete_dup_normalize_title(value):
    return " ".join(str(value or "").split())


def delete_dup_normalize_path(value):
    text = str(value or "").strip().replace("\\", "/").replace(" / ", "/")
    return " / ".join(part.strip() for part in text.split("/") if part.strip())


def delete_dup_leaf_paths(album):
    paths = {
        delete_dup_normalize_path(folder.get("path") or folder.get("title"))
        for folder in (album.get("folders") or {}).values()
        if folder.get("path") or folder.get("title")
    }
    paths.discard("")
    if not paths:
        return ("[ROOT_ALBUMS]",)
    return tuple(sorted(
        path for path in paths
        if not any(other != path and other.startswith(path + " / ") for other in paths)
    ))


def delete_dup_asset_identity(asset):
    unique_id = asset.get("photo_library_asset_unique_id")
    if unique_id is None:
        return None
    return json.dumps(unique_id, ensure_ascii=False, sort_keys=True, default=str)


def delete_dup_checksum(values):
    payload = "\n".join(sorted(values))
    return hashlib.sha256(payload.encode("utf-8")).hexdigest()[:16]


def delete_dup_tsv(value):
    return str(value if value is not None else "-").replace("\t", " ").replace("\r", " ").replace("\n", " ")


def delete_dup_album_records(inventory):
    cross_members = defaultdict(set)
    local_uuid_members = defaultdict(set)
    missing_identity_albums = set()

    for asset in inventory.get("assets") or []:
        identity = delete_dup_asset_identity(asset)
        local_uuid = str(asset.get("uuid") or "")
        for album_uuid in (asset.get("albums") or {}):
            album_uuid = str(album_uuid)
            if identity is None:
                missing_identity_albums.add(album_uuid)
            else:
                cross_members[album_uuid].add(identity)
            if local_uuid:
                local_uuid_members[album_uuid].add(local_uuid)

    records = []
    for inventory_key, album in (inventory.get("albums") or {}).items():
        raw_title = str(album.get("title") or "")
        normalized_title = delete_dup_normalize_title(raw_title)
        if not normalized_title:
            continue

        album_uuid = str(album.get("uuid") or inventory_key)
        member_set = frozenset(cross_members.get(album_uuid, set()))
        local_uuid_set = frozenset(local_uuid_members.get(album_uuid, set()))

        records.append({
            "album_uuid": album_uuid,
            "raw_title": raw_title,
            "normalized_title": normalized_title,
            "folder_paths": delete_dup_leaf_paths(album),
            "member_set": member_set,
            "local_uuid_set": local_uuid_set,
            "asset_count": len(local_uuid_set),
            "membership_checksum": delete_dup_checksum(local_uuid_set),
            "has_missing_asset_identity": album_uuid in missing_identity_albums,
        })

    return records


current_records = delete_dup_album_records(inventory_current)
backup_records = delete_dup_album_records(inventory_backup)

backup_by_identity = defaultdict(list)
for row in backup_records:
    backup_by_identity[(row["normalized_title"], row["member_set"])].append(row)

current_by_identity = defaultdict(list)
for row in current_records:
    current_by_identity[(row["normalized_title"], row["member_set"])].append(row)

review_rows = []
manifest_rows = []
candidate_group_number = 0

for identity_key, current_group in sorted(
    current_by_identity.items(),
    key=lambda item: item[0][0],
):
    normalized_title, member_set = identity_key
    if len(current_group) < 2:
        continue

    candidate_group_number += 1
    backup_group = backup_by_identity.get(identity_key, [])
    raw_titles = {row["raw_title"] for row in current_group}
    title_match_type = (
        "EXACT_TITLE" if len(raw_titles) == 1
        else "WHITESPACE_ONLY_DIFFERENCE"
    )

    decision = "SKIP"
    reason = ""
    keeper = None
    delete_rows = []

    if normalized_title in DELETE_DUP_EXCLUDED_TITLES:
        reason = "EXCLUDED_TITLE"
    elif not member_set:
        reason = "EMPTY_MEMBERSHIP"
    elif any(row["has_missing_asset_identity"] for row in current_group):
        reason = "CURRENT_ASSET_IDENTITY_MISSING"
    elif len({row["local_uuid_set"] for row in current_group}) != 1:
        reason = "CURRENT_LOCAL_MEMBERSHIP_MISMATCH"
    elif len(backup_group) != 1:
        reason = f"BACKUP_MATCH_COUNT_{len(backup_group)}"
    else:
        backup_album = backup_group[0]

        if backup_album["has_missing_asset_identity"]:
            reason = "BACKUP_ASSET_IDENTITY_MISSING"
        elif len(backup_album["folder_paths"]) != 1:
            reason = "BACKUP_PATH_AMBIGUOUS"
        else:
            canonical_path = backup_album["folder_paths"][0]
            keepers = [
                row for row in current_group
                if row["folder_paths"] == (canonical_path,)
                and row["raw_title"] == backup_album["raw_title"]
            ]

            if len(keepers) != 1:
                reason = f"CURRENT_KEEPER_COUNT_{len(keepers)}"
            else:
                keeper = keepers[0]
                delete_rows = [
                    row for row in current_group
                    if row["album_uuid"] != keeper["album_uuid"]
                ]

                if not delete_rows:
                    reason = "NO_EXTRA_OBJECT"
                elif any(len(row["folder_paths"]) != 1 for row in delete_rows):
                    reason = "DELETE_PATH_AMBIGUOUS"
                elif any(row["folder_paths"][0] == canonical_path for row in delete_rows):
                    reason = "EXTRA_OBJECT_AT_CANONICAL_PATH"
                else:
                    decision = "READY_TO_DELETE"
                    reason = "STRICT_BACKUP_CANONICAL_MATCH"

    canonical_path = (
        backup_group[0]["folder_paths"][0]
        if len(backup_group) == 1 and len(backup_group[0]["folder_paths"]) == 1
        else "-"
    )
    
    if (
        decision == "READY_TO_DELETE"
        and canonical_path != DELETE_DUP_TARGET_KEEP_FOLDER_PATH
    ):
        decision = "SKIP"
        reason = "OUTSIDE_TARGET_KEEP_FOLDER_PATH"
        keeper = None
        delete_rows = []

    for row in current_group:
        role = (
            "KEEP" if keeper and row["album_uuid"] == keeper["album_uuid"]
            else "DELETE" if decision == "READY_TO_DELETE"
            else "REVIEW"
        )

        review_rows.append({
            "candidate_group_number": candidate_group_number,
            "decision": decision,
            "reason": reason,
            "role": role,
            "title_match_type": title_match_type,
            "folder_path": " || ".join(row["folder_paths"]),
            "backup_canonical_path": canonical_path,
            "album_title": row["raw_title"],
            "album_uuid": row["album_uuid"],
            "asset_count": row["asset_count"],
            "membership_checksum": row["membership_checksum"],
            "current_group_object_count": len(current_group),
            "backup_matching_album_count": len(backup_group),
        })

    if decision == "READY_TO_DELETE":
        for row in delete_rows:
            manifest_rows.append({
                "operation": "delete_duplicate_album",
                "delete_album_uuid": row["album_uuid"],
                "keep_album_uuid": keeper["album_uuid"],
                "delete_folder_path": row["folder_paths"][0],
                "keep_folder_path": canonical_path,
                "album_title": row["raw_title"],
                "normalized_title": normalized_title,
                "asset_count": row["asset_count"],
                "membership_checksum": row["membership_checksum"],
                "title_match_type": title_match_type,
                "backup_canonical_path": canonical_path,
                "candidate_group_number": candidate_group_number,
                "status": "READY_TO_DELETE",
            })


DELETE_DUP_INBOX.mkdir(parents=True, exist_ok=True)
DELETE_DUP_ARCHIVE.mkdir(parents=True, exist_ok=True)

review_fields = [
    "candidate_group_number", "decision", "reason", "role",
    "title_match_type", "folder_path", "backup_canonical_path",
    "album_title", "album_uuid", "asset_count", "membership_checksum",
    "current_group_object_count", "backup_matching_album_count",
]

with DELETE_DUP_REVIEW.open("w", encoding="utf-8", newline="") as file:
    writer = csv.DictWriter(file, fieldnames=review_fields, delimiter="\t", lineterminator="\n")
    writer.writeheader()
    for row in review_rows:
        writer.writerow({key: delete_dup_tsv(row.get(key)) for key in review_fields})

manifest_fields = [
    "operation", "delete_album_uuid", "keep_album_uuid",
    "delete_folder_path", "keep_folder_path", "album_title",
    "normalized_title", "asset_count", "membership_checksum",
    "title_match_type", "backup_canonical_path",
    "candidate_group_number", "status",
]

with DELETE_DUP_MANIFEST.open("w", encoding="utf-8", newline="") as file:
    writer = csv.DictWriter(file, fieldnames=manifest_fields, delimiter="\t", lineterminator="\n")
    for row in manifest_rows:
        writer.writerow({key: delete_dup_tsv(row.get(key)) for key in manifest_fields})

archive_review = None
archive_manifest = None
if WRITE_DELETE_DUP_ARCHIVE:
    timestamp = datetime.now().strftime("%Y%m%d_%H%M%S_%f")
    archive_review = DELETE_DUP_ARCHIVE / f"{timestamp}__delete_duplicate_candidates.tsv"
    archive_manifest = DELETE_DUP_ARCHIVE / f"{timestamp}__delete_duplicate_manifest.tsv"
    shutil.copy2(DELETE_DUP_REVIEW, archive_review)
    shutil.copy2(DELETE_DUP_MANIFEST, archive_manifest)

print("=" * 100)
print("Delete duplicate album manifest exported")
print("=" * 100)
print("Strict delete rows:", len(manifest_rows))
print("Candidate groups reviewed:", candidate_group_number)
print("Review TSV:", DELETE_DUP_REVIEW)
print("Executable manifest:", DELETE_DUP_MANIFEST)
if archive_review:
    print("Archive review:", archive_review)
    print("Archive manifest:", archive_manifest)
print()
print("Manifest fixed column order, no header:")
print("\t".join(manifest_fields))


Delete duplicate album manifest exported
Strict delete rows: 0
Candidate groups reviewed: 9
Review TSV: /Users/huohsien/Downloads/PhotosRepairMVP_Inbox/DeleteDuplicateAlbumsCandidatesLatest.tsv
Executable manifest: /Users/huohsien/Downloads/PhotosRepairMVP_Inbox/DeleteDuplicateAlbumsManifestLatest.tsv
Archive review: /Users/huohsien/Downloads/PhotosRepairMVP_Inbox/archive/20260617_155536_373314__delete_duplicate_candidates.tsv
Archive manifest: /Users/huohsien/Downloads/PhotosRepairMVP_Inbox/archive/20260617_155536_373314__delete_duplicate_manifest.tsv

Manifest fixed column order, no header:
operation	delete_album_uuid	keep_album_uuid	delete_folder_path	keep_folder_path	album_title	normalized_title	asset_count	membership_checksum	title_match_type	backup_canonical_path	candidate_group_number	status


In [3]:
# ============================================================
# Post-delete audit — verify deleted albums absent
# and keeper albums intact
#
# READ ONLY:
# - reads the existing executed delete manifest
# - checks the freshly rebuilt inventory_current
# - does not modify Photos
# - does not overwrite the delete manifest
# ============================================================

from collections import defaultdict
from pathlib import Path
import csv
import hashlib


AUDIT_MANIFEST_PATH = (
    Path.home()
    / "Downloads"
    / "PhotosRepairMVP_Inbox"
    / "DeleteDuplicateAlbumsManifestLatest.tsv"
)

AUDIT_OUTPUT_PATH = (
    Path.home()
    / "Downloads"
    / "PhotosRepairMVP_Inbox"
    / "DeleteDuplicateAlbumsPostDeleteAuditLatest.tsv"
)


MANIFEST_FIELDS = [
    "operation",
    "delete_album_uuid",
    "keep_album_uuid",
    "delete_folder_path",
    "keep_folder_path",
    "album_title",
    "normalized_title",
    "asset_count",
    "membership_checksum",
    "title_match_type",
    "backup_canonical_path",
    "candidate_group_number",
    "status",
]


def audit_normalize_title(value):
    return " ".join(str(value or "").split())


def audit_normalize_path(value):
    text = (
        str(value or "")
        .strip()
        .replace("\\", "/")
        .replace(" / ", "/")
    )

    return " / ".join(
        part.strip()
        for part in text.split("/")
        if part.strip()
    )


def audit_leaf_paths(album):
    paths = {
        audit_normalize_path(
            folder.get("path")
            or folder.get("title")
        )
        for folder in (
            album.get("folders") or {}
        ).values()
        if folder.get("path")
        or folder.get("title")
    }

    paths.discard("")

    if not paths:
        return ("[ROOT_ALBUMS]",)

    return tuple(
        sorted(
            path
            for path in paths
            if not any(
                other != path
                and other.startswith(path + " / ")
                for other in paths
            )
        )
    )


def audit_checksum(values):
    payload = "\n".join(sorted(values))

    return hashlib.sha256(
        payload.encode("utf-8")
    ).hexdigest()[:16]


# ------------------------------------------------------------
# Read the executed manifest.
# It intentionally has no header.
# ------------------------------------------------------------

manifest_rows = []

with AUDIT_MANIFEST_PATH.open(
    "r",
    encoding="utf-8",
    newline="",
) as file:
    reader = csv.DictReader(
        file,
        fieldnames=MANIFEST_FIELDS,
        delimiter="\t",
    )

    manifest_rows = [
        dict(row)
        for row in reader
        if any(
            str(value or "").strip()
            for value in row.values()
        )
    ]


# ------------------------------------------------------------
# Build live album and membership indexes from the freshly
# rebuilt inventory_current.
# ------------------------------------------------------------

albums_by_uuid = {}

for inventory_key, album in (
    inventory_current.get("albums") or {}
).items():
    album_uuid = str(
        album.get("uuid")
        or inventory_key
    )

    albums_by_uuid[album_uuid] = album


members_by_album_uuid = defaultdict(set)

for asset in (
    inventory_current.get("assets") or []
):
    asset_uuid = str(
        asset.get("uuid") or ""
    )

    if not asset_uuid:
        continue

    for album_uuid in (
        asset.get("albums") or {}
    ):
        members_by_album_uuid[
            str(album_uuid)
        ].add(asset_uuid)


# ------------------------------------------------------------
# Verify every executed manifest job.
# ------------------------------------------------------------

audit_rows = []

for job_number, manifest_row in enumerate(
    manifest_rows,
    start=1,
):
    delete_uuid = str(
        manifest_row["delete_album_uuid"]
    )

    keep_uuid = str(
        manifest_row["keep_album_uuid"]
    )

    expected_keep_path = (
        audit_normalize_path(
            manifest_row["keep_folder_path"]
        )
    )

    expected_normalized_title = (
        audit_normalize_title(
            manifest_row["normalized_title"]
        )
    )

    expected_asset_count = int(
        manifest_row["asset_count"]
    )

    expected_checksum = str(
        manifest_row["membership_checksum"]
    )

    delete_album = albums_by_uuid.get(
        delete_uuid
    )

    keep_album = albums_by_uuid.get(
        keep_uuid
    )

    delete_exists = (
        delete_album is not None
    )

    delete_member_refs = (
        members_by_album_uuid.get(
            delete_uuid,
            set(),
        )
    )

    delete_has_asset_references = bool(
        delete_member_refs
    )

    keep_exists = (
        keep_album is not None
    )

    if keep_exists:
        actual_keep_paths = (
            audit_leaf_paths(keep_album)
        )

        actual_keep_title = str(
            keep_album.get("title") or ""
        )

        actual_normalized_title = (
            audit_normalize_title(
                actual_keep_title
            )
        )

        actual_members = (
            members_by_album_uuid.get(
                keep_uuid,
                set(),
            )
        )

        actual_asset_count = len(
            actual_members
        )

        actual_checksum = audit_checksum(
            actual_members
        )
    else:
        actual_keep_paths = tuple()
        actual_keep_title = ""
        actual_normalized_title = ""
        actual_asset_count = -1
        actual_checksum = "-"

    keep_path_ok = (
        keep_exists
        and actual_keep_paths
        == (expected_keep_path,)
    )

    keep_title_ok = (
        keep_exists
        and actual_normalized_title
        == expected_normalized_title
    )

    keep_asset_count_ok = (
        keep_exists
        and actual_asset_count
        == expected_asset_count
    )

    keep_checksum_ok = (
        keep_exists
        and actual_checksum
        == expected_checksum
    )

    delete_absent_ok = (
        not delete_exists
    )

    delete_relationships_absent_ok = (
        not delete_has_asset_references
    )

    failure_reasons = []

    if not delete_absent_ok:
        failure_reasons.append(
            "DELETE_UUID_STILL_EXISTS"
        )

    if not delete_relationships_absent_ok:
        failure_reasons.append(
            "DELETE_UUID_STILL_REFERENCED_BY_ASSETS"
        )

    if not keep_exists:
        failure_reasons.append(
            "KEEPER_UUID_MISSING"
        )
    else:
        if not keep_path_ok:
            failure_reasons.append(
                "KEEPER_PATH_CHANGED"
            )

        if not keep_title_ok:
            failure_reasons.append(
                "KEEPER_TITLE_CHANGED"
            )

        if not keep_asset_count_ok:
            failure_reasons.append(
                "KEEPER_ASSET_COUNT_CHANGED"
            )

        if not keep_checksum_ok:
            failure_reasons.append(
                "KEEPER_MEMBERSHIP_CHANGED"
            )

    fully_passed = (
        not failure_reasons
    )

    audit_rows.append({
        "job_number": job_number,
        "candidate_group_number": (
            manifest_row[
                "candidate_group_number"
            ]
        ),
        "audit_status": (
            "PASS"
            if fully_passed
            else "FAIL"
        ),
        "failure_reasons": (
            " || ".join(failure_reasons)
            if failure_reasons
            else "-"
        ),
        "delete_album_uuid": delete_uuid,
        "delete_uuid_absent": (
            delete_absent_ok
        ),
        "delete_asset_relationships_absent": (
            delete_relationships_absent_ok
        ),
        "keep_album_uuid": keep_uuid,
        "keeper_exists": keep_exists,
        "expected_keep_folder_path": (
            expected_keep_path
        ),
        "actual_keep_folder_paths": (
            " || ".join(actual_keep_paths)
            if actual_keep_paths
            else "-"
        ),
        "keeper_path_ok": keep_path_ok,
        "keeper_title_ok": keep_title_ok,
        "expected_asset_count": (
            expected_asset_count
        ),
        "actual_asset_count": (
            actual_asset_count
        ),
        "keeper_asset_count_ok": (
            keep_asset_count_ok
        ),
        "expected_membership_checksum": (
            expected_checksum
        ),
        "actual_membership_checksum": (
            actual_checksum
        ),
        "keeper_membership_checksum_ok": (
            keep_checksum_ok
        ),
        "album_title": (
            manifest_row["album_title"]
        ),
    })


# ------------------------------------------------------------
# Export a readable audit TSV.
# ------------------------------------------------------------

AUDIT_FIELDS = [
    "job_number",
    "candidate_group_number",
    "audit_status",
    "failure_reasons",
    "delete_album_uuid",
    "delete_uuid_absent",
    "delete_asset_relationships_absent",
    "keep_album_uuid",
    "keeper_exists",
    "expected_keep_folder_path",
    "actual_keep_folder_paths",
    "keeper_path_ok",
    "keeper_title_ok",
    "expected_asset_count",
    "actual_asset_count",
    "keeper_asset_count_ok",
    "expected_membership_checksum",
    "actual_membership_checksum",
    "keeper_membership_checksum_ok",
    "album_title",
]

with AUDIT_OUTPUT_PATH.open(
    "w",
    encoding="utf-8",
    newline="",
) as file:
    writer = csv.DictWriter(
        file,
        fieldnames=AUDIT_FIELDS,
        delimiter="\t",
        lineterminator="\n",
    )

    writer.writeheader()
    writer.writerows(audit_rows)


# ------------------------------------------------------------
# Summary.
# ------------------------------------------------------------

passed_rows = [
    row
    for row in audit_rows
    if row["audit_status"] == "PASS"
]

failed_rows = [
    row
    for row in audit_rows
    if row["audit_status"] == "FAIL"
]


print("=" * 100)
print("Delete Duplicate Albums — Post-delete Audit")
print("=" * 100)
print("Manifest jobs:", len(audit_rows))
print("Fully passed:", len(passed_rows))
print("Failed:", len(failed_rows))
print()

print(
    "Delete UUID absent:",
    sum(
        bool(row["delete_uuid_absent"])
        for row in audit_rows
    ),
)

print(
    "Delete relationships absent:",
    sum(
        bool(
            row[
                "delete_asset_relationships_absent"
            ]
        )
        for row in audit_rows
    ),
)

print(
    "Keepers present:",
    sum(
        bool(row["keeper_exists"])
        for row in audit_rows
    ),
)

print(
    "Keeper paths correct:",
    sum(
        bool(row["keeper_path_ok"])
        for row in audit_rows
    ),
)

print(
    "Keeper titles correct:",
    sum(
        bool(row["keeper_title_ok"])
        for row in audit_rows
    ),
)

print(
    "Keeper asset counts correct:",
    sum(
        bool(
            row["keeper_asset_count_ok"]
        )
        for row in audit_rows
    ),
)

print(
    "Keeper membership checksums correct:",
    sum(
        bool(
            row[
                "keeper_membership_checksum_ok"
            ]
        )
        for row in audit_rows
    ),
)

print()
print("Audit TSV:", AUDIT_OUTPUT_PATH)

if failed_rows:
    print()
    print("FAILED JOBS:")

    for row in failed_rows:
        print(
            row["job_number"],
            row["failure_reasons"],
            row["delete_album_uuid"],
            row["keep_album_uuid"],
        )
else:
    print()
    print("SUCCESS: all manifest jobs passed post-delete audit.")

Delete Duplicate Albums — Post-delete Audit
Manifest jobs: 34
Fully passed: 34
Failed: 0

Delete UUID absent: 34
Delete relationships absent: 34
Keepers present: 34
Keeper paths correct: 34
Keeper titles correct: 34
Keeper asset counts correct: 34
Keeper membership checksums correct: 34

Audit TSV: /Users/huohsien/Downloads/PhotosRepairMVP_Inbox/DeleteDuplicateAlbumsPostDeleteAuditLatest.tsv

SUCCESS: all manifest jobs passed post-delete audit.


In [ ]:
# ============================================================
# Cross-library inventory diff summary report
# ============================================================

diff_records, inventory_diff_summary_report = write_cross_library_inventory_diff_summary_report(
    inventory_backup=inventory_backup,
    inventory_current=inventory_current,
    report_root=Path("reports/test2_inventory_diff_summary"),
    label="snapshot",
)

{
    "inventory_diff_summary_report": inventory_diff_summary_report,
}

In [ ]:
# ============================================================
# Status: ASSET_MISSING_FROM_CURRENT — DONE 20260615
# ============================================================

missing_current_review_result = print_missing_current_review(
    diff_records=diff_records,
    inventory_current=inventory_current,
    max_current_candidates=5,
)

# Optional: write text/TSV files only when you explicitly want files.
WRITE_MISSING_CURRENT_REVIEW_REPORT_FILES = False

if WRITE_MISSING_CURRENT_REVIEW_REPORT_FILES:
    missing_current_review_file_result = write_missing_current_review_report(
        diff_records=diff_records,
        inventory_current=inventory_current,
        output_dir=Path("reports/test2_missing_current_review"),
        report_name_prefix="asset_missing_from_current_review",
    )
    print_missing_current_review_report_summary(missing_current_review_file_result)


In [ ]:
# =======================================================================
# Repare folder-album relations in current default using info in backup:
#   Find the info needed for reparments
# =======================================================================

from pathlib import Path
from datetime import datetime
import csv
import re
import shutil


# ------------------------------------------------------------
# User target
# ------------------------------------------------------------

TARGET_PARENT_FOLDER_PATH = "股票"

# ------------------------------------------------------------
# Path helpers
# ------------------------------------------------------------

def _normalize_photo_folder_path(path):
    if path is None:
        return ""

    text = str(path).strip()
    text = text.replace("\\", "/")
    text = text.replace(" / ", "/")

    parts = [
        part.strip()
        for part in text.split("/")
        if part.strip()
    ]

    return " / ".join(parts)


def _join_photo_path(folder_path, album_title):
    folder = _normalize_photo_folder_path(folder_path)
    album = str(album_title).strip()

    if folder:
        return f"{folder} / {album}"

    return album


def _deepest_folder_path_for_album(album):
    folders = album.get("folders") or {}

    folder_paths = [
        _normalize_photo_folder_path(folder.get("path") or folder.get("title"))
        for folder in folders.values()
        if folder.get("path") or folder.get("title")
    ]

    folder_paths = [
        path
        for path in folder_paths
        if path
    ]

    if not folder_paths:
        return ""

    return sorted(
        folder_paths,
        key=lambda value: (
            value.count(" / "),
            len(value),
        ),
    )[-1]


def _album_target_from_album(album):
    album_title = album.get("title")

    if not album_title:
        return None

    folder_path = _deepest_folder_path_for_album(album)
    album_title = str(album_title).strip()
    album_path = _join_photo_path(folder_path, album_title)

    return {
        "folder_path": folder_path,
        "album_title": album_title,
        "album_path": album_path,
    }


def _asset_album_targets(asset):
    targets = []

    for album in (asset.get("albums") or {}).values():
        target = _album_target_from_album(album)

        if target:
            targets.append(target)

    deduped = {}
    for target in targets:
        key = (
            target["folder_path"],
            target["album_title"],
            target["album_path"],
        )
        deduped[key] = target

    return list(deduped.values())


def _album_path_is_under_folder(album_path, parent_folder_path):
    parent = _normalize_photo_folder_path(parent_folder_path)

    return (
        album_path == parent
        or album_path.startswith(parent + " / ")
    )


def discover_backup_album_targets_under_folder(
    inventory_backup,
    target_parent_folder_path,
):
    target_parent = _normalize_photo_folder_path(target_parent_folder_path)

    rows_by_album_path = {}

    for asset in inventory_backup.get("assets") or []:
        for target in _asset_album_targets(asset):
            album_path = target["album_path"]

            if not _album_path_is_under_folder(album_path, target_parent):
                continue

            row = rows_by_album_path.setdefault(
                album_path,
                {
                    "folder_path": target["folder_path"],
                    "album_title": target["album_title"],
                    "album_path": target["album_path"],
                    "backup_assets": 0,
                    "backup_photos": 0,
                    "backup_videos": 0,
                },
            )

            row["backup_assets"] += 1

            if asset.get("is_movie"):
                row["backup_videos"] += 1
            else:
                row["backup_photos"] += 1

    rows = list(rows_by_album_path.values())

    rows.sort(
        key=lambda row: (
            -row["backup_assets"],
            row["album_path"],
        )
    )

    return rows


backup_album_targets = discover_backup_album_targets_under_folder(
    inventory_backup=inventory_backup,
    target_parent_folder_path=TARGET_PARENT_FOLDER_PATH,
)

print("=" * 120)
print("Backup album targets under folder:", _normalize_photo_folder_path(TARGET_PARENT_FOLDER_PATH))
print("=" * 120)
print("backup album target count:", len(backup_album_targets))
print()

print(
    "{:>3}  {:>13}  {:>13}  {:>13}  {}".format(
        "idx",
        "backup_assets",
        "backup_photos",
        "backup_videos",
        "backup_album_path",
    )
)
print(
    "{:>3}  {:>13}  {:>13}  {:>13}  {}".format(
        "---",
        "-------------",
        "-------------",
        "-------------",
        "-" * 80,
    )
)

for index, row in enumerate(backup_album_targets, start=1):
    print(
        "{:>3}  {:>13}  {:>13}  {:>13}  {}".format(
            index,
            row["backup_assets"],
            row["backup_photos"],
            row["backup_videos"],
            row["album_path"],
        )
    )

In [ ]:
# =======================================================================
# Preflight check:
#   Are target Backup albums already present elsewhere in Current Default?
# =======================================================================
#
# Purpose:
#   Before exporting TSV and running PhotosRepairMVP, check whether albums under
#   TARGET_PARENT_FOLDER_PATH in Backup already exist in Current Default by
#   the same album title, but under a different folder/root path.
#
# This cell:
#   - reads inventory_backup / inventory_current only
#   - does not write to Photos Library
#   - does not export PhotoKit repair TSV
#   - writes one small .txt report for review
#   - returns one short dict

from collections import defaultdict, Counter
from pathlib import Path
from datetime import datetime
import re


def _preflight_safe_filename_part(text):
    text = str(text).strip()
    text = text.replace(" / ", "__")
    text = text.replace("/", "__")
    text = re.sub(r"[^A-Za-z0-9._-]+", "_", text)
    text = re.sub(r"_+", "_", text)
    return text.strip("_") or "untitled"


PREFLIGHT_TARGET_PARENT_FOLDER_PATH = TARGET_PARENT_FOLDER_PATH
PREFLIGHT_REPORT_ROOT = Path(
    "reports/test2_scattered_current_album_preflight"
)
PREFLIGHT_MAX_PREVIEW_ROWS = 30


def _preflight_asset_unique_id_key(asset):
    unique_id = asset.get(
        "photo_library_asset_unique_id"
    )

    if unique_id is None:
        return None

    return tuple(unique_id)


def _preflight_asset_album_paths(asset):
    album_paths = []

    for album in (
        asset.get("albums") or {}
    ).values():
        target = _album_target_from_album(album)

        if target:
            album_paths.append(
                target["album_path"]
            )

    return tuple(
        sorted(set(album_paths))
    )


def _preflight_asset_in_album_path(
    asset,
    album_path,
):
    return album_path in (
        _preflight_asset_album_paths(asset)
    )


def _preflight_album_full_path(album):
    target = _album_target_from_album(album)

    if target is None:
        return None

    return target["album_path"]


def _preflight_root_folder(album_path):
    parts = [
        part.strip()
        for part in str(album_path).split(" / ")
        if part.strip()
    ]

    if len(parts) <= 1:
        return "[ROOT_ALBUMS]"

    return parts[0]


def _preflight_build_current_album_members_by_uuid(
    inventory_current,
):
    members_by_album_uuid = defaultdict(set)

    for asset in (
        inventory_current.get("assets") or []
    ):
        asset_key = (
            _preflight_asset_unique_id_key(asset)
        )

        if asset_key is None:
            continue

        for album_uuid in (
            asset.get("albums") or {}
        ):
            members_by_album_uuid[
                album_uuid
            ].add(asset_key)

    return members_by_album_uuid


def _preflight_build_current_albums_by_title(
    inventory_current,
):
    albums_by_title = defaultdict(list)

    for album in (
        inventory_current.get("albums") or {}
    ).values():
        title = str(
            album.get("title") or ""
        ).strip()

        if not title:
            continue

        albums_by_title[title].append(album)

    return albums_by_title


def _preflight_backup_members_for_album_path(
    inventory_backup,
    album_path,
):
    members = set()

    for asset in (
        inventory_backup.get("assets") or []
    ):
        if not _preflight_asset_in_album_path(
            asset,
            album_path,
        ):
            continue

        asset_key = (
            _preflight_asset_unique_id_key(asset)
        )

        if asset_key is not None:
            members.add(asset_key)

    return members


def check_target_backup_albums_scattered_in_current(
    *,
    inventory_backup,
    inventory_current,
    backup_album_targets,
    report_root,
    label,
):
    report_root = Path(report_root)

    report_root.mkdir(
        parents=True,
        exist_ok=True,
    )

    run_time = datetime.now()

    report_dir = (
        report_root
        / f"{run_time:%Y%m%d-%H%M%S}__{label}"
    )

    report_dir.mkdir(
        parents=True,
        exist_ok=False,
    )

    report_path = (
        report_dir
        / "scattered_current_album_preflight.txt"
    )

    current_albums_by_title = (
        _preflight_build_current_albums_by_title(
            inventory_current
        )
    )

    current_members_by_album_uuid = (
        _preflight_build_current_album_members_by_uuid(
            inventory_current
        )
    )

    rows = []

    for index, target in enumerate(
        backup_album_targets,
        start=1,
    ):
        backup_album_path = target[
            "album_path"
        ]

        backup_album_title = target[
            "album_title"
        ]

        backup_members = (
            _preflight_backup_members_for_album_path(
                inventory_backup,
                backup_album_path,
            )
        )

        current_candidates = (
            current_albums_by_title.get(
                backup_album_title,
                [],
            )
        )

        candidate_summaries = []

        for current_album in current_candidates:
            current_album_uuid = (
                current_album.get("uuid")
            )

            current_album_path = (
                _preflight_album_full_path(
                    current_album
                )
            )

            current_members = (
                current_members_by_album_uuid.get(
                    current_album_uuid,
                    set(),
                )
            )

            overlap = (
                backup_members
                & current_members
            )

            candidate_summaries.append({
                "current_album_uuid":
                    current_album_uuid,

                "current_album_path":
                    current_album_path,

                "current_root_folder":
                    _preflight_root_folder(
                        current_album_path
                    ),

                "current_asset_count":
                    len(current_members),

                "overlap_asset_count":
                    len(overlap),

                "missing_from_candidate_count":
                    len(
                        backup_members
                        - current_members
                    ),

                "extra_in_candidate_count":
                    len(
                        current_members
                        - backup_members
                    ),
            })

        candidate_summaries.sort(
            key=lambda row: (
                -row[
                    "overlap_asset_count"
                ],
                row[
                    "current_album_path"
                ] or "",
            )
        )

        best_candidate = (
            candidate_summaries[0]
            if candidate_summaries
            else None
        )

        if not candidate_summaries:
            status = (
                "NO_SAME_TITLE_CURRENT_ALBUM"
            )

        elif (
            best_candidate[
                "current_album_path"
            ]
            == backup_album_path
        ):
            status = (
                "ALREADY_AT_TARGET_PATH"
            )

        elif (
            best_candidate[
                "overlap_asset_count"
            ]
            == len(backup_members)
            and
            best_candidate[
                "current_asset_count"
            ]
            == len(backup_members)
        ):
            status = (
                "SAME_TITLE_ELSEWHERE_"
                "FULL_MEMBER_MATCH"
            )

        elif (
            best_candidate[
                "overlap_asset_count"
            ]
            > 0
        ):
            status = (
                "SAME_TITLE_ELSEWHERE_"
                "PARTIAL_MEMBER_OVERLAP"
            )

        else:
            status = (
                "SAME_TITLE_ELSEWHERE_"
                "NO_MEMBER_OVERLAP"
            )

        rows.append({
            "idx":
                index,

            "status":
                status,

            "backup_album_path":
                backup_album_path,

            "backup_album_title":
                backup_album_title,

            "backup_asset_count":
                len(backup_members),

            "current_same_title_album_count":
                len(candidate_summaries),

            "best_current_album_path":
                (
                    best_candidate[
                        "current_album_path"
                    ]
                    if best_candidate
                    else ""
                ),

            "best_current_root_folder":
                (
                    best_candidate[
                        "current_root_folder"
                    ]
                    if best_candidate
                    else ""
                ),

            "best_current_asset_count":
                (
                    best_candidate[
                        "current_asset_count"
                    ]
                    if best_candidate
                    else 0
                ),

            "best_overlap_asset_count":
                (
                    best_candidate[
                        "overlap_asset_count"
                    ]
                    if best_candidate
                    else 0
                ),

            "best_missing_from_candidate_count":
                (
                    best_candidate[
                        "missing_from_candidate_count"
                    ]
                    if best_candidate
                    else len(backup_members)
                ),

            "best_extra_in_candidate_count":
                (
                    best_candidate[
                        "extra_in_candidate_count"
                    ]
                    if best_candidate
                    else 0
                ),
        })

    status_counts = Counter(
        row["status"]
        for row in rows
    )

    root_counts = Counter(
        row["best_current_root_folder"]
        for row in rows
        if row[
            "best_current_root_folder"
        ]
    )

    with report_path.open(
        "w",
        encoding="utf-8",
    ) as file:
        file.write(
            "Scattered Current album "
            "preflight check\n"
        )

        file.write(
            "=" * 120 + "\n"
        )

        file.write(
            f"label\t{label}\n"
        )

        file.write(
            "run_timestamp\t"
            f"{run_time.isoformat(timespec='seconds')}"
            "\n"
        )

        file.write(
            "target_parent_folder_path\t"
            f"{PREFLIGHT_TARGET_PARENT_FOLDER_PATH}"
            "\n"
        )

        file.write(
            "backup_album_target_count\t"
            f"{len(backup_album_targets)}"
            "\n"
        )

        file.write("\n")
        file.write("status_counts\n")

        for status, count in (
            status_counts.most_common()
        ):
            file.write(
                f"{status}\t{count}\n"
            )

        file.write("\n")

        file.write(
            "best_current_root_folder_counts\n"
        )

        for root, count in (
            root_counts.most_common()
        ):
            file.write(
                f"{root}\t{count}\n"
            )

        file.write("\n")

        file.write(
            "\t".join([
                "idx",
                "status",
                "backup_asset_count",
                "current_same_title_album_count",
                "best_current_asset_count",
                "best_overlap_asset_count",
                "best_missing_from_candidate_count",
                "best_extra_in_candidate_count",
                "best_current_root_folder",
                "best_current_album_path",
                "backup_album_path",
            ])
            + "\n"
        )

        for row in rows:
            file.write(
                "\t".join([
                    str(row["idx"]),

                    row["status"],

                    str(
                        row[
                            "backup_asset_count"
                        ]
                    ),

                    str(
                        row[
                            "current_same_title_album_count"
                        ]
                    ),

                    str(
                        row[
                            "best_current_asset_count"
                        ]
                    ),

                    str(
                        row[
                            "best_overlap_asset_count"
                        ]
                    ),

                    str(
                        row[
                            "best_missing_from_candidate_count"
                        ]
                    ),

                    str(
                        row[
                            "best_extra_in_candidate_count"
                        ]
                    ),

                    row[
                        "best_current_root_folder"
                    ],

                    row[
                        "best_current_album_path"
                    ],

                    row[
                        "backup_album_path"
                    ],
                ])
                + "\n"
            )

    preview_rows = rows[
        :PREFLIGHT_MAX_PREVIEW_ROWS
    ]

    return {
        "target_parent_folder_path":
            PREFLIGHT_TARGET_PARENT_FOLDER_PATH,

        "backup_album_target_count":
            len(backup_album_targets),

        "status_counts":
            dict(status_counts),

        "best_current_root_folder_counts":
            dict(root_counts),

        "report_path":
            str(report_path),

        "preview_rows":
            preview_rows,
    }

preflight_label = (
    "target_"
    + _preflight_safe_filename_part(
        PREFLIGHT_TARGET_PARENT_FOLDER_PATH
    )
)

scattered_current_album_preflight_result = (
    check_target_backup_albums_scattered_in_current(
        inventory_backup=inventory_backup,
        inventory_current=inventory_current,
        backup_album_targets=backup_album_targets,
        report_root=PREFLIGHT_REPORT_ROOT,
        label=preflight_label,
    )
)

scattered_current_album_preflight_result

In [ ]:
# =======================================================================================
# Repare folder-album relations in current default using info in backup:
#   Export TSV manifest for PhotosRepairMVP to actually repair them in Photots Liberary
# =======================================================================================
from pathlib import Path
from datetime import datetime
import csv
import re
import shutil


# ------------------------------------------------------------
# Output protocol
# ------------------------------------------------------------

REPAIR_INBOX_DIR = Path.home() / "Downloads" / "PhotosRepairMVP_Inbox"
LATEST_MANIFEST_PATH = REPAIR_INBOX_DIR / "RepairManifestLatest.tsv"

WRITE_ARCHIVE_COPY = True
ARCHIVE_DIR = REPAIR_INBOX_DIR / "archive"


# Test mode:
#   TARGET_START_INDEX = 0 means use the first row from backup_album_targets.
#   MAX_ALBUMS_TO_EXPORT = 1 means export only one album.
#
# Batch mode:
#   Set MAX_ALBUMS_TO_EXPORT = None to export all remaining targets.
TARGET_START_INDEX = 0
MAX_ALBUMS_TO_EXPORT = None


# ------------------------------------------------------------
# Asset / TSV helpers
# ------------------------------------------------------------

def _safe_filename_part(text):
    text = str(text).strip()
    text = text.replace(" / ", "__")
    text = text.replace("/", "__")
    text = re.sub(r"[^A-Za-z0-9._-]+", "_", text)
    text = re.sub(r"_+", "_", text)
    return text.strip("_") or "untitled"


def _asset_album_paths(asset):
    album_paths = []

    for album in (asset.get("albums") or {}).values():
        target = _album_target_from_album(album)

        if target:
            album_paths.append(target["album_path"])

    return tuple(sorted(set(album_paths)))


def _asset_in_album_path(asset, album_path):
    return album_path in _asset_album_paths(asset)


def _asset_unique_id_key(asset):
    unique_id = asset.get("photo_library_asset_unique_id")

    if unique_id is None:
        return None

    return tuple(unique_id)


def _format_dt(value):
    if value is None:
        return "-"

    if hasattr(value, "strftime"):
        return value.strftime("%Y-%m-%d %H:%M:%S")

    text = str(value)
    text = text.replace("T", " ")

    if "+" in text:
        text = text.split("+", 1)[0]

    return text[:19]


def _asset_media_label(asset):
    return "video" if asset.get("is_movie") else "photo"


def _asset_original_size(asset):
    width = asset.get("original_width")
    height = asset.get("original_height")

    if width is None or height is None:
        return "-"

    return f"{width}x{height}"


def _tsv_clean(value):
    if value is None:
        return "-"

    text = str(value)
    text = text.replace("\t", " ")
    text = text.replace("\r", " ")
    text = text.replace("\n", " ")
    return text


def _build_current_asset_index_by_unique_id(inventory_current):
    index = {}

    for asset in inventory_current.get("assets") or []:
        key = _asset_unique_id_key(asset)

        if key is None:
            continue

        if key in index:
            raise RuntimeError(f"Duplicate current unique_id: {key}")

        index[key] = asset

    return index


def build_repair_rows_for_album_target(
    *,
    inventory_backup,
    current_by_unique_id,
    folder_path,
    album_title,
):
    normalized_folder_path = _normalize_photo_folder_path(folder_path)
    album_path = _join_photo_path(normalized_folder_path, album_title)

    backup_assets_in_album = [
        asset
        for asset in inventory_backup.get("assets") or []
        if _asset_in_album_path(asset, album_path)
    ]

    backup_assets_in_album.sort(
        key=lambda asset: (
            _format_dt(asset.get("date")),
            asset.get("original_filename") or "",
            asset.get("uuid") or "",
        )
    )

    repair_rows = []

    for backup_asset in backup_assets_in_album:
        key = _asset_unique_id_key(backup_asset)
        current_asset = current_by_unique_id.get(key) if key is not None else None

        repair_rows.append({
            "operation": "add_asset_to_album",
            "folder_path": normalized_folder_path,
            "album_title": album_title,
            "media": _asset_media_label(backup_asset),
            "original_filename": backup_asset.get("original_filename") or "-",
            "date": _format_dt(backup_asset.get("date")),
            "original_size": _asset_original_size(backup_asset),
            "backup_uuid": backup_asset.get("uuid") or "-",
            "current_uuid": current_asset.get("uuid") if current_asset else "-",
            "status": "MATCHED_IN_CURRENT" if current_asset else "MISSING_IN_CURRENT",
        })

    return repair_rows


def export_repair_manifest_for_album_targets(
    *,
    inventory_backup,
    inventory_current,
    album_targets,
    latest_manifest_path,
    write_archive_copy=True,
    archive_dir=None,
):
    if not album_targets:
        raise RuntimeError("No album targets to export.")

    latest_manifest_path = Path(latest_manifest_path)

    if archive_dir is None:
        archive_dir = latest_manifest_path.parent / "archive"
    else:
        archive_dir = Path(archive_dir)

    current_by_unique_id = _build_current_asset_index_by_unique_id(
        inventory_current
    )

    all_repair_rows = []
    album_summaries = []

    for target_index, target in enumerate(album_targets, start=1):
        folder_path = target["folder_path"]
        album_title = target["album_title"]
        album_path = target["album_path"]

        repair_rows = build_repair_rows_for_album_target(
            inventory_backup=inventory_backup,
            current_by_unique_id=current_by_unique_id,
            folder_path=folder_path,
            album_title=album_title,
        )

        photo_count = sum(row["media"] == "photo" for row in repair_rows)
        video_count = sum(row["media"] == "video" for row in repair_rows)
        matched_count = sum(row["status"] == "MATCHED_IN_CURRENT" for row in repair_rows)
        missing_count = sum(row["status"] == "MISSING_IN_CURRENT" for row in repair_rows)

        album_summaries.append({
            "idx": target_index,
            "folder_path": folder_path,
            "album_title": album_title,
            "album_path": album_path,
            "repair_rows": len(repair_rows),
            "photos": photo_count,
            "videos": video_count,
            "matched": matched_count,
            "missing": missing_count,
        })

        all_repair_rows.extend(repair_rows)

    manifest_fields = [
        "operation",
        "folder_path",
        "album_title",
        "media",
        "original_filename",
        "date",
        "original_size",
        "backup_uuid",
        "current_uuid",
        "status",
    ]

    latest_manifest_path.parent.mkdir(parents=True, exist_ok=True)

    with latest_manifest_path.open("w", encoding="utf-8", newline="") as file:
        writer = csv.DictWriter(
            file,
            fieldnames=manifest_fields,
            delimiter="\t",
            lineterminator="\n",
            extrasaction="ignore",
        )

        # Header intentionally omitted.
        # PhotosRepairMVP reads this TSV by fixed column order.

        for row in all_repair_rows:
            writer.writerow({
                key: _tsv_clean(row.get(key))
                for key in manifest_fields
            })

    archive_path = None

    if write_archive_copy:
        archive_dir.mkdir(parents=True, exist_ok=True)

        timestamp = datetime.now().strftime("%Y%m%d_%H%M%S_%f")

        archive_filename = (
            f"{timestamp}__"
            f"album_targets_{len(album_targets)}.tsv"
        )

        archive_path = archive_dir / archive_filename
        shutil.copy2(latest_manifest_path, archive_path)

    total_matched = sum(
        row["status"] == "MATCHED_IN_CURRENT"
        for row in all_repair_rows
    )

    total_missing = sum(
        row["status"] == "MISSING_IN_CURRENT"
        for row in all_repair_rows
    )

    print("=" * 120)
    print("PhotosRepairMVP repair manifest exported")
    print("=" * 120)
    print("album targets exported:", len(album_targets))
    print("total TSV rows:", len(all_repair_rows))
    print("matched rows:", total_matched)
    print("missing rows:", total_missing)
    print()
    print("latest path:", latest_manifest_path)

    if archive_path:
        print("archive path:", archive_path)

    print()
    print("Manifest protocol, fixed column order, no header row:")
    print("\t".join(manifest_fields))
    print()
    print("Album summaries:")
    print(
        "{:>3}  {:>7}  {:>7}  {:>7}  {:>7}  {}".format(
            "idx",
            "rows",
            "photos",
            "videos",
            "missing",
            "album_path",
        )
    )
    print(
        "{:>3}  {:>7}  {:>7}  {:>7}  {:>7}  {}".format(
            "---",
            "-------",
            "-------",
            "-------",
            "-------",
            "-" * 80,
        )
    )

    for summary in album_summaries:
        print(
            "{:>3}  {:>7}  {:>7}  {:>7}  {:>7}  {}".format(
                summary["idx"],
                summary["repair_rows"],
                summary["photos"],
                summary["videos"],
                summary["missing"],
                summary["album_path"],
            )
        )

    return {
        "album_targets": album_targets,
        "album_summaries": album_summaries,
        "repair_rows": all_repair_rows,
        "latest_manifest_path": latest_manifest_path,
        "archive_path": archive_path,
    }


if MAX_ALBUMS_TO_EXPORT is None:
    selected_album_targets = backup_album_targets[TARGET_START_INDEX:]
else:
    selected_album_targets = backup_album_targets[
        TARGET_START_INDEX:TARGET_START_INDEX + MAX_ALBUMS_TO_EXPORT
    ]

repair_manifest_export_result = export_repair_manifest_for_album_targets(
    inventory_backup=inventory_backup,
    inventory_current=inventory_current,
    album_targets=selected_album_targets,
    latest_manifest_path=LATEST_MANIFEST_PATH,
    write_archive_copy=WRITE_ARCHIVE_COPY,
    archive_dir=ARCHIVE_DIR,
)

In [ ]:
# ============================================================
# Optional helper: Backup album source snapshot under one folder
# ============================================================
#
# Purpose:
#   Print a human-readable list of Backup albums under one selected folder.
#
# This helper:
#   - uses Backup only
#   - does not compare Current
#   - does not verify repair results
#   - does not export a PhotoKit repair manifest
#   - is not used by later cells
#
# Suggested use:
#   Run once, copy the output to a separate text file, then clear the output.
# ============================================================

TARGET_PARENT_FOLDER_PATH = "NSFW"


def optional_album_snapshot_normalize_folder_path(path):
    if path is None:
        return ""

    text = str(path).strip()
    text = text.replace("\\", "/")
    text = text.replace(" / ", "/")

    parts = []

    for part in text.split("/"):
        part = part.strip()

        if part:
            parts.append(part)

    return " / ".join(parts)


def optional_album_snapshot_album_folder_path(album):
    folders = album.get("folders") or {}
    paths = []

    for folder in folders.values():
        path = folder.get("path") or folder.get("title")
        path = optional_album_snapshot_normalize_folder_path(path)

        if path:
            paths.append(path)

    if not paths:
        return ""

    return sorted(
        paths,
        key=lambda value: (
            value.count(" / "),
            len(value),
        ),
    )[-1]


def optional_album_snapshot_album_path(album):
    title = album.get("title")

    if not title:
        return None

    folder_path = optional_album_snapshot_album_folder_path(album)

    if folder_path:
        return folder_path + " / " + str(title)

    return str(title)


def optional_album_snapshot_asset_album_paths(asset):
    result = []

    for album in (asset.get("albums") or {}).values():
        path = optional_album_snapshot_album_path(album)

        if path:
            result.append(path)

    return tuple(sorted(set(result)))


def optional_album_snapshot_is_under_target(album_path, target_parent):
    return (
        album_path == target_parent
        or album_path.startswith(target_parent + " / ")
    )


target_parent = optional_album_snapshot_normalize_folder_path(
    TARGET_PARENT_FOLDER_PATH
)

target_album_rows_by_path = {}

for asset in inventory_backup.get("assets") or []:
    for album_path in optional_album_snapshot_asset_album_paths(asset):
        if not optional_album_snapshot_is_under_target(album_path, target_parent):
            continue

        row = target_album_rows_by_path.setdefault(
            album_path,
            {
                "album_path": album_path,
                "assets": 0,
                "photos": 0,
                "videos": 0,
            },
        )

        row["assets"] += 1

        if asset.get("is_movie"):
            row["videos"] += 1
        else:
            row["photos"] += 1

target_album_rows = list(target_album_rows_by_path.values())

target_album_rows.sort(
    key=lambda row: (
        -row["assets"],
        row["album_path"],
    )
)

output_lines = []

output_lines.append("=" * 120)
output_lines.append(
    "Backup album source snapshot under folder: {}".format(target_parent)
)
output_lines.append("=" * 120)
output_lines.append("")
output_lines.append("Purpose:")
output_lines.append("  Backup-only album source list for manual reference.")
output_lines.append("  It does not compare Current.")
output_lines.append("  It does not verify repair results.")
output_lines.append("  It does not export a PhotoKit repair manifest.")
output_lines.append("")
output_lines.append("backup album count: {}".format(len(target_album_rows)))
output_lines.append("")
output_lines.append(
    "{:>3}  {:>13}  {:>13}  {:>13}  {}".format(
        "idx",
        "backup_assets",
        "backup_photos",
        "backup_videos",
        "backup_album_path",
    )
)
output_lines.append(
    "{:>3}  {:>13}  {:>13}  {:>13}  {}".format(
        "---",
        "-------------",
        "-------------",
        "-------------",
        "-" * 80,
    )
)

for index, row in enumerate(target_album_rows, start=1):
    output_lines.append(
        "{:>3}  {:>13}  {:>13}  {:>13}  {}".format(
            index,
            row["assets"],
            row["photos"],
            row["videos"],
            row["album_path"],
        )
    )

print("\n".join(output_lines))

In [ ]:
# =====================================================================
# Check: Backup vs Current album paths and asset counts, whole library
# =====================================================================

from collections import defaultdict, Counter
import hashlib


# None = whole library.
# Or set to "NSFW", "股票", etc. to limit to one root folder.
TARGET_ROOT_FOLDER_PATH = None


def _albumcmp_normalize_folder_path(path):
    if path is None:
        return ""

    text = str(path).strip()
    text = text.replace("\\", "/")
    text = text.replace(" / ", "/")

    parts = [
        part.strip()
        for part in text.split("/")
        if part.strip()
    ]

    return " / ".join(parts)


def _albumcmp_join_photo_path(folder_path, album_title):
    folder = _albumcmp_normalize_folder_path(folder_path)
    album = str(album_title).strip()

    if folder:
        return f"{folder} / {album}"

    return album


def _albumcmp_deepest_folder_path_for_album(album):
    folders = album.get("folders") or {}

    folder_paths = [
        _albumcmp_normalize_folder_path(folder.get("path") or folder.get("title"))
        for folder in folders.values()
        if folder.get("path") or folder.get("title")
    ]

    folder_paths = [
        path
        for path in folder_paths
        if path
    ]

    if not folder_paths:
        return ""

    return sorted(
        folder_paths,
        key=lambda value: (
            value.count(" / "),
            len(value),
        ),
    )[-1]


def _albumcmp_album_full_path(album):
    album_title = album.get("title")

    if not album_title:
        return None

    folder_path = _albumcmp_deepest_folder_path_for_album(album)

    return _albumcmp_join_photo_path(
        folder_path,
        str(album_title).strip(),
    )


def _albumcmp_is_under_root(album_path, root_folder_path):
    if root_folder_path is None:
        return True

    root = _albumcmp_normalize_folder_path(root_folder_path)

    if not root:
        return True

    return (
        album_path == root
        or album_path.startswith(root + " / ")
    )


def _albumcmp_asset_key(asset):
    unique_id = asset.get("photo_library_asset_unique_id")

    if unique_id is not None:
        return "unique_id:" + repr(tuple(unique_id))

    uuid = asset.get("uuid")

    if uuid:
        return "uuid:" + str(uuid)

    return None


def _albumcmp_checksum(member_set):
    joined = "\n".join(sorted(member_set))
    return hashlib.sha256(joined.encode("utf-8")).hexdigest()[:16]


def _albumcmp_root_folder_from_album_path(album_path):
    parts = [
        part.strip()
        for part in str(album_path or "").split(" / ")
        if part.strip()
    ]

    if len(parts) <= 1:
        return "[ROOT_ALBUMS]"

    return parts[0]


def _albumcmp_collect_album_catalog(inventory, root_folder_path=None):
    album_rows = []
    album_path_counts = Counter()

    for album in (inventory.get("albums") or {}).values():
        album_path = _albumcmp_album_full_path(album)

        if not album_path:
            continue

        if not _albumcmp_is_under_root(album_path, root_folder_path):
            continue

        folder_path = _albumcmp_deepest_folder_path_for_album(album)
        album_title = str(album.get("title") or "").strip()

        album_path_counts[album_path] += 1

        album_rows.append({
            "folder_path": folder_path,
            "album_title": album_title,
            "album_path": album_path,
            "root_folder": _albumcmp_root_folder_from_album_path(album_path),
            "album_uuid": album.get("uuid") or "-",
        })

    album_rows.sort(
        key=lambda row: (
            row["album_path"],
            row["album_uuid"],
        )
    )

    return album_rows, album_path_counts


def _albumcmp_collect_asset_members_by_album_path(inventory, root_folder_path=None):
    members_by_album_path = defaultdict(set)
    no_album_asset_keys = set()

    for asset in inventory.get("assets") or []:
        asset_key = _albumcmp_asset_key(asset)

        if asset_key is None:
            continue

        asset_albums = asset.get("albums") or {}

        if not asset_albums:
            no_album_asset_keys.add(asset_key)
            continue

        for album in asset_albums.values():
            album_path = _albumcmp_album_full_path(album)

            if not album_path:
                continue

            if not _albumcmp_is_under_root(album_path, root_folder_path):
                continue

            members_by_album_path[album_path].add(asset_key)

    return members_by_album_path, no_album_asset_keys


backup_album_catalog, backup_album_path_counts = _albumcmp_collect_album_catalog(
    inventory_backup,
    TARGET_ROOT_FOLDER_PATH,
)

current_album_catalog, current_album_path_counts = _albumcmp_collect_album_catalog(
    inventory_current,
    TARGET_ROOT_FOLDER_PATH,
)

backup_members_by_album_path, backup_no_album_assets = _albumcmp_collect_asset_members_by_album_path(
    inventory_backup,
    TARGET_ROOT_FOLDER_PATH,
)

current_members_by_album_path, current_no_album_assets = _albumcmp_collect_asset_members_by_album_path(
    inventory_current,
    TARGET_ROOT_FOLDER_PATH,
)

backup_album_paths = set(row["album_path"] for row in backup_album_catalog)
current_album_paths = set(row["album_path"] for row in current_album_catalog)

all_album_paths = sorted(backup_album_paths | current_album_paths)

comparison_rows = []

for album_path in all_album_paths:
    backup_members = backup_members_by_album_path.get(album_path, set())
    current_members = current_members_by_album_path.get(album_path, set())

    missing_members = backup_members - current_members
    extra_members = current_members - backup_members

    backup_album_exists = album_path in backup_album_paths
    current_album_exists = album_path in current_album_paths

    if backup_album_exists and not current_album_exists:
        status = "MISSING_ALBUM_IN_CURRENT"
    elif current_album_exists and not backup_album_exists:
        status = "CURRENT_ONLY_ALBUM"
    elif not missing_members and not extra_members:
        status = "OK"
    else:
        status = "MEMBERSHIP_DIFF"

    comparison_rows.append({
        "status": status,
        "root_folder": _albumcmp_root_folder_from_album_path(album_path),
        "album_path": album_path,
        "backup_album_objects": backup_album_path_counts.get(album_path, 0),
        "current_album_objects": current_album_path_counts.get(album_path, 0),
        "backup_assets": len(backup_members),
        "current_assets": len(current_members),
        "missing_assets_in_current": len(missing_members),
        "extra_assets_in_current": len(extra_members),
        "backup_checksum": _albumcmp_checksum(backup_members),
        "current_checksum": _albumcmp_checksum(current_members),
        "checksum_match": _albumcmp_checksum(backup_members) == _albumcmp_checksum(current_members),
    })


target_label = (
    "whole library"
    if TARGET_ROOT_FOLDER_PATH is None
    else _albumcmp_normalize_folder_path(TARGET_ROOT_FOLDER_PATH)
)

print("=" * 180)
print("Backup vs Current album paths and asset counts:", target_label)
print("=" * 180)
print()

print("album catalog counts")
print("-" * 180)
print("backup album objects:       ", len(backup_album_catalog))
print("current album objects:      ", len(current_album_catalog))
print("backup unique album paths:  ", len(backup_album_paths))
print("current unique album paths: ", len(current_album_paths))
print("all unique album paths:     ", len(all_album_paths))
print()

print("no-album asset counts")
print("-" * 180)
print("backup no-album assets: ", len(backup_no_album_assets))
print("current no-album assets:", len(current_no_album_assets))
print()

status_counts = Counter(row["status"] for row in comparison_rows)

print("status counts")
print("-" * 180)

for status, count in sorted(status_counts.items()):
    print(f"{status}: {count}")

print()
print("root folder breakdown")
print("-" * 180)
print(
    "{:<50} {:>10} {:>10} {:>10} {:>10} {:>10}".format(
        "root_folder",
        "backup",
        "current",
        "missing",
        "new",
        "diff",
    )
)

all_root_folders = sorted(
    set(row["root_folder"] for row in comparison_rows)
)

for root_folder in all_root_folders:
    rows = [
        row
        for row in comparison_rows
        if row["root_folder"] == root_folder
    ]

    backup_count = sum(1 for row in rows if row["backup_album_objects"] > 0)
    current_count = sum(1 for row in rows if row["current_album_objects"] > 0)
    missing_count = sum(1 for row in rows if row["status"] == "MISSING_ALBUM_IN_CURRENT")
    new_count = sum(1 for row in rows if row["status"] == "CURRENT_ONLY_ALBUM")
    diff_count = sum(1 for row in rows if row["status"] == "MEMBERSHIP_DIFF")

    print(
        "{:<50} {:>10} {:>10} {:>10} {:>10} {:>10}".format(
            root_folder[:50],
            backup_count,
            current_count,
            missing_count,
            new_count,
            diff_count,
        )
    )

print()
print("missing / current-only album paths")
print("-" * 180)

for row in comparison_rows:
    if row["status"] in {"MISSING_ALBUM_IN_CURRENT", "CURRENT_ONLY_ALBUM"}:
        print(
            f"{row['status']} | "
            f"root={row['root_folder']} | "
            f"backup_album_objects={row['backup_album_objects']} | "
            f"current_album_objects={row['current_album_objects']} | "
            f"backup_assets={row['backup_assets']} | "
            f"current_assets={row['current_assets']} | "
            f"{row['album_path']}"
        )

print()
print("membership differences")
print("-" * 180)

for row in comparison_rows:
    if row["status"] == "MEMBERSHIP_DIFF":
        print(
            f"{row['status']} | "
            f"root={row['root_folder']} | "
            f"backup_assets={row['backup_assets']} | "
            f"current_assets={row['current_assets']} | "
            f"missing_assets_in_current={row['missing_assets_in_current']} | "
            f"extra_assets_in_current={row['extra_assets_in_current']} | "
            f"{row['album_path']}"
        )

print()
print("duplicate full album paths")
print("-" * 180)

duplicate_found = False

for album_path, count in sorted(backup_album_path_counts.items()):
    if count > 1:
        duplicate_found = True
        print(f"BACKUP duplicate path x{count}: {album_path}")

for album_path, count in sorted(current_album_path_counts.items()):
    if count > 1:
        duplicate_found = True
        print(f"CURRENT duplicate path x{count}: {album_path}")

if not duplicate_found:
    print("-")

print()
print("full table")
print("-" * 180)
print(
    "{:<24} {:<28} {:>7} {:>7} {:>8} {:>8} {:>8} {:>8} {:>18} {:>18}  {}".format(
        "status",
        "root_folder",
        "b_alb",
        "c_alb",
        "b_assets",
        "c_assets",
        "missing",
        "extra",
        "backup_checksum",
        "current_checksum",
        "album_path",
    )
)

for row in comparison_rows:
    print(
        "{:<24} {:<28} {:>7} {:>7} {:>8} {:>8} {:>8} {:>8} {:>18} {:>18}  {}".format(
            row["status"],
            row["root_folder"][:28],
            row["backup_album_objects"],
            row["current_album_objects"],
            row["backup_assets"],
            row["current_assets"],
            row["missing_assets_in_current"],
            row["extra_assets_in_current"],
            row["backup_checksum"],
            row["current_checksum"],
            row["album_path"],
        )
    )

# Test Playground 

In [ ]:
# ============================================================
# TEMP — Verify Job 1 duplicate deletion
#
# READ ONLY:
# - reads the live Current Default Photos Library database
# - does not use inventory_current cache
# - does not modify Photos Library
# ============================================================

from pathlib import Path
import json
import osxphotos


OLD_ROOT_ALBUM_UUID = "C0F17467-F7FD-4346-8F99-121D7DD1E9DD"
NEW_STOCK_ALBUM_UUID = "205EB2D8-3689-41F5-B6D7-7E8506505B38"
EXPECTED_ASSET_COUNT = 168


# Read the saved Current Default Photos Library path.
history_path = Path(
    "data/local_config/test2_library_paths.json"
)

library_history = json.loads(
    history_path.read_text(encoding="utf-8")
)

current_library_path = Path(
    library_history["current_default"]
)

print("Current Library:", current_library_path)
print()


# Read current album objects directly from the live database.
photosdb = osxphotos.PhotosDB(
    str(current_library_path)
)

albums_by_uuid = {
    str(album.uuid): album
    for album in photosdb.album_info
}


def inspect_album(label, album_uuid):
    album = albums_by_uuid.get(album_uuid)

    print("=" * 80)
    print(label)
    print("=" * 80)

    if album is None:
        print("Album exists: NO")
        print("UUID:", album_uuid)
        print()
        return None

    folder_names = list(
        album.folder_names or []
    )

    folder_path = (
        " / ".join(folder_names)
        if folder_names
        else "[ROOT_ALBUMS]"
    )

    print("Album exists: YES")
    print("UUID:", album.uuid)
    print("Folder Path:", folder_path)
    print("Asset Count:", len(album.photos))
    print("Title Length:", len(album.title))
    print("Title Ending:", repr(album.title[-40:]))
    print()

    return {
        "folder_path": folder_path,
        "asset_count": len(album.photos),
        "title": album.title,
    }


old_root = inspect_album(
    "OLD ROOT ALBUM",
    OLD_ROOT_ALBUM_UUID,
)

new_stock = inspect_album(
    "NEW 股票 ALBUM",
    NEW_STOCK_ALBUM_UUID,
)


root_deleted_ok = old_root is None

stock_exists_ok = new_stock is not None

stock_path_ok = (
    new_stock is not None
    and new_stock["folder_path"] == "股票"
)

stock_count_ok = (
    new_stock is not None
    and new_stock["asset_count"] == EXPECTED_ASSET_COUNT
)


print("=" * 80)
print("FINAL VERIFICATION")
print("=" * 80)
print(
    "Root duplicate deleted:",
    "PASS" if root_deleted_ok else "FAIL",
)
print(
    "股票 album remains:",
    "PASS" if stock_exists_ok else "FAIL",
)
print(
    "股票 folder path correct:",
    "PASS" if stock_path_ok else "FAIL",
)
print(
    f"股票 asset count is {EXPECTED_ASSET_COUNT}:",
    "PASS" if stock_count_ok else "FAIL",
)
print()

all_ok = (
    root_deleted_ok
    and stock_exists_ok
    and stock_path_ok
    and stock_count_ok
)

print(
    "OVERALL RESULT:",
    "SUCCESS" if all_ok else "NOT YET CORRECT",
)

# OLD Stuffs

In [ ]:
# # ============================================================
# # Appendix A: Duplicate diagnostic archive
# # 
# # TEMP: Diagnose potential duplicate groups by SHA256
# #       with full manual-review metadata
# # ============================================================

# import hashlib
# import os
# import time
# from datetime import datetime


# def compute_sha256_for_asset(asset, chunk_size=1024 * 1024):
#     cached_sha256 = asset.get("content_sha256")
#     if cached_sha256:
#         return cached_sha256

#     path = asset.get("path")

#     if path is None:
#         return None

#     if not os.path.exists(path):
#         return None

#     sha256 = hashlib.sha256()

#     with open(path, "rb") as f:
#         while True:
#             chunk = f.read(chunk_size)

#             if not chunk:
#                 break

#             sha256.update(chunk)

#     digest = sha256.hexdigest()
#     asset["content_sha256"] = digest
#     return digest


# def build_potential_duplicate_groups_by_unique_id(inventory):
#     unique_id_to_assets = {}

#     for asset in inventory["assets"]:
#         unique_id = asset.get("photo_library_asset_unique_id")

#         if unique_id is None:
#             continue

#         if unique_id not in unique_id_to_assets:
#             unique_id_to_assets[unique_id] = []

#         unique_id_to_assets[unique_id].append(asset)

#     return {
#         unique_id: assets
#         for unique_id, assets in unique_id_to_assets.items()
#         if len(assets) > 1
#     }


# def normalize_string_list(values):
#     result = []

#     if values is None:
#         return result

#     if isinstance(values, str):
#         return [values]

#     if isinstance(values, dict):
#         iterable = values.values()
#     elif isinstance(values, (list, tuple, set)):
#         iterable = values
#     else:
#         return [str(values)]

#     for item in iterable:
#         if item is None:
#             continue

#         if isinstance(item, str):
#             value = item
#         elif isinstance(item, dict):
#             value = (
#                 item.get("title")
#                 or item.get("name")
#                 or item.get("path")
#                 or item.get("folder_path")
#                 or item.get("album_path")
#             )
#         else:
#             value = str(item)

#         if value:
#             result.append(value)

#     return sorted(set(result))


# def get_asset_album_titles(asset):
#     albums = asset.get("albums")
#     return normalize_string_list(albums)


# def get_asset_folder_paths(asset):
#     folders = asset.get("folders")
#     folder_paths = normalize_string_list(folders)

#     # Some inventory formats may store folder paths under different keys.
#     extra_candidates = [
#         asset.get("folder_paths"),
#         asset.get("folder_path"),
#         asset.get("album_folder_paths"),
#     ]

#     for candidate in extra_candidates:
#         folder_paths.extend(normalize_string_list(candidate))

#     return sorted(set(folder_paths))


# def get_asset_keywords(asset):
#     keywords = asset.get("keywords")
#     return normalize_string_list(keywords)


# def get_asset_description(asset):
#     return (
#         asset.get("description")
#         or asset.get("caption")
#         or asset.get("title")
#         or ""
#     )


# def parse_date_added_for_sort(asset):
#     date_added = asset.get("date_added")

#     if not date_added:
#         return datetime.max

#     if isinstance(date_added, datetime):
#         return date_added

#     text = str(date_added)

#     try:
#         return datetime.fromisoformat(text.replace("Z", "+00:00"))
#     except Exception:
#         return datetime.max


# def asset_metadata_signature(asset):
#     return {
#         "albums": tuple(get_asset_album_titles(asset)),
#         "folders": tuple(get_asset_folder_paths(asset)),
#         "keywords": tuple(get_asset_keywords(asset)),
#         "description": get_asset_description(asset),
#         "favorite": asset.get("favorite"),
#         "hidden": asset.get("hidden"),
#         "hasadjustments": asset.get("hasadjustments"),
#         "adjustment_signature": asset.get("adjustment_signature"),
#     }


# def metadata_score(asset):
#     return (
#         len(get_asset_album_titles(asset)) * 10
#         + len(get_asset_folder_paths(asset)) * 10
#         + len(get_asset_keywords(asset)) * 5
#         + (1 if get_asset_description(asset) else 0)
#         + (1 if asset.get("favorite") else 0)
#         + (1 if asset.get("hidden") else 0)
#     )


# def choose_representative_asset(assets):
#     # Prefer metadata-rich assets; tie-break by earliest Date Added.
#     return sorted(
#         assets,
#         key=lambda asset: (
#             -metadata_score(asset),
#             parse_date_added_for_sort(asset),
#             asset.get("uuid") or "",
#         ),
#     )[0]


# def print_asset_manual_review_block(asset, indent="  "):
#     print(f"{indent}UUID:", asset.get("uuid"))
#     print(f"{indent}Original File Name:", asset.get("original_filename"))
#     print(f"{indent}Filename:", asset.get("filename"))
#     print(f"{indent}Date:", asset.get("date"))
#     print(f"{indent}Date Added:", asset.get("date_added"))
#     print(f"{indent}File Size:", asset.get("file_size_bytes"))
#     print(f"{indent}Has Adjustments:", asset.get("hasadjustments"))
#     print(f"{indent}Adjustment Signature:", asset.get("adjustment_signature"))
#     print(f"{indent}Width x Height:", asset.get("width"), "x", asset.get("height"))
#     print(f"{indent}Original Width x Height:", asset.get("original_width"), "x", asset.get("original_height"))
#     print(f"{indent}Albums:", get_asset_album_titles(asset))
#     print(f"{indent}Folder Paths:", get_asset_folder_paths(asset))
#     print(f"{indent}Keywords:", get_asset_keywords(asset))
#     print(f"{indent}Description:", get_asset_description(asset))
#     print(f"{indent}Favorite:", asset.get("favorite"))
#     print(f"{indent}Hidden:", asset.get("hidden"))
#     print(f"{indent}Path:", asset.get("path"))


# def print_cleanup_recommendation(assets):
#     metadata_signatures = [asset_metadata_signature(asset) for asset in assets]
#     metadata_all_same = all(
#         signature == metadata_signatures[0]
#         for signature in metadata_signatures
#     )

#     representative = choose_representative_asset(assets)

#     if metadata_all_same:
#         print("Recommendation:")
#         print("  Metadata appears identical.")
#         print("  Keep earliest / representative asset:")
#         print("   ", representative.get("uuid"))
#         print("  Delete other duplicate asset(s):")
#         for asset in assets:
#             if asset is not representative:
#                 print("   ", asset.get("uuid"))
#     else:
#         print("Recommendation:")
#         print("  Metadata differs across duplicate assets.")
#         print("  Do NOT blindly delete.")
#         print("  Suggested representative, based on richer metadata + earliest Date Added:")
#         print("   ", representative.get("uuid"))
#         print("  Before deleting others, manually confirm whether album/folder/keyword membership should be preserved.")


# def diagnose_potential_duplicate_groups_with_sha256_and_metadata(
#     inventory,
#     label,
#     max_true_duplicate_groups_to_print=50,
#     max_key_collision_groups_to_print=20,
# ):
#     start_time = time.perf_counter()

#     potential_groups = build_potential_duplicate_groups_by_unique_id(inventory)

#     true_content_duplicate_groups = []
#     key_collision_groups = []
#     sha_error_assets = []

#     checked_asset_count = 0

#     for unique_id, assets in potential_groups.items():
#         sha256_to_assets = {}

#         for asset in assets:
#             checked_asset_count += 1
#             sha256 = compute_sha256_for_asset(asset)

#             if sha256 is None:
#                 sha_error_assets.append(asset)
#                 continue

#             if sha256 not in sha256_to_assets:
#                 sha256_to_assets[sha256] = []

#             sha256_to_assets[sha256].append(asset)

#         duplicate_sha_groups = {
#             sha256: sha_assets
#             for sha256, sha_assets in sha256_to_assets.items()
#             if len(sha_assets) > 1
#         }

#         if duplicate_sha_groups:
#             for sha256, sha_assets in duplicate_sha_groups.items():
#                 true_content_duplicate_groups.append(
#                     {
#                         "unique_id": unique_id,
#                         "sha256": sha256,
#                         "assets": sha_assets,
#                     }
#                 )

#         if len(sha256_to_assets) > 1:
#             key_collision_groups.append(
#                 {
#                     "unique_id": unique_id,
#                     "sha256_to_assets": sha256_to_assets,
#                 }
#             )

#     elapsed = time.perf_counter() - start_time

#     print(label)
#     print("-" * 120)
#     print("potential duplicate unique_id group count:", len(potential_groups))
#     print("checked asset count:", checked_asset_count)
#     print("sha error asset count:", len(sha_error_assets))
#     print("true content duplicate group count:", len(true_content_duplicate_groups))
#     print("key collision group count:", len(key_collision_groups))
#     print("elapsed seconds:", round(elapsed, 3))

#     print()
#     print("TRUE CONTENT DUPLICATE GROUPS — MANUAL REVIEW")
#     print("-" * 120)

#     for index, group in enumerate(true_content_duplicate_groups, start=1):
#         if index > max_true_duplicate_groups_to_print:
#             print("... more true content duplicate groups not printed")
#             break

#         assets_sorted = sorted(
#             group["assets"],
#             key=lambda asset: (
#                 parse_date_added_for_sort(asset),
#                 asset.get("uuid") or "",
#             ),
#         )

#         print("=" * 120)
#         print(f"Group {index:02d}")
#         print("=" * 120)
#         print("unique_id:", group["unique_id"])
#         print("sha256:", group["sha256"])
#         print("asset count:", len(assets_sorted))

#         first_asset = assets_sorted[0]
#         print("Original File Name:", first_asset.get("original_filename"))
#         print("Date:", first_asset.get("date"))
#         print("File Size:", first_asset.get("file_size_bytes"))
#         print("Adjustment Signature:", first_asset.get("adjustment_signature"))

#         union_albums = sorted(
#             set(
#                 album
#                 for asset in assets_sorted
#                 for album in get_asset_album_titles(asset)
#             )
#         )
#         union_folders = sorted(
#             set(
#                 folder
#                 for asset in assets_sorted
#                 for folder in get_asset_folder_paths(asset)
#             )
#         )
#         union_keywords = sorted(
#             set(
#                 keyword
#                 for asset in assets_sorted
#                 for keyword in get_asset_keywords(asset)
#             )
#         )

#         print("Union Albums:", union_albums)
#         print("Union Folder Paths:", union_folders)
#         print("Union Keywords:", union_keywords)

#         print()
#         print_cleanup_recommendation(assets_sorted)
#         print()

#         for asset_index, asset in enumerate(assets_sorted, start=1):
#             print("-" * 120)
#             print(f"Asset {asset_index}")
#             print_asset_manual_review_block(asset, indent="  ")

#         print()

#     print()
#     print("KEY COLLISION GROUPS")
#     print("-" * 120)

#     for index, group in enumerate(key_collision_groups, start=1):
#         if index > max_key_collision_groups_to_print:
#             print("... more key collision groups not printed")
#             break

#         print("=" * 120)
#         print(f"Key Collision Group {index:02d}")
#         print("=" * 120)
#         print("unique_id:", group["unique_id"])
#         print("sha256 count:", len(group["sha256_to_assets"]))

#         for sha256, assets in group["sha256_to_assets"].items():
#             print("  sha256:", sha256)
#             print("  asset count:", len(assets))

#             for asset in assets:
#                 print("    uuid:", asset.get("uuid"))
#                 print("    original_filename:", asset.get("original_filename"))
#                 print("    filename:", asset.get("filename"))
#                 print("    date:", asset.get("date"))
#                 print("    date_added:", asset.get("date_added"))
#                 print("    file_size_bytes:", asset.get("file_size_bytes"))
#                 print("    albums:", get_asset_album_titles(asset))
#                 print("    folder_paths:", get_asset_folder_paths(asset))
#                 print("    keywords:", get_asset_keywords(asset))
#                 print("    path:", asset.get("path"))

#         print()

#     return {
#         "potential_groups": potential_groups,
#         "true_content_duplicate_groups": true_content_duplicate_groups,
#         "key_collision_groups": key_collision_groups,
#         "sha_error_assets": sha_error_assets,
#     }


# backup_duplicate_diagnostic = diagnose_potential_duplicate_groups_with_sha256_and_metadata(
#     inventory_backup,
#     "BACKUP potential duplicate diagnostic with metadata",
# )

# print()

# current_duplicate_diagnostic = diagnose_potential_duplicate_groups_with_sha256_and_metadata(
#     inventory_current,
#     "CURRENT potential duplicate diagnostic with metadata",
# )

In [ ]:
# # ============================================================
# # Appendix B: Debug assets without unique ID
# # 
# # DEBUG: Dump assets without photo_library_asset_unique_id
# # ============================================================

# import os
# import time
# from collections import Counter

# def debug_dump_assets_without_photo_library_asset_unique_id(inventory, label, max_print=80):
#     missing_assets = [
#         asset
#         for asset in inventory["assets"]
#         if asset.get("photo_library_asset_unique_id") is None
#     ]

#     reason_counter = Counter()

#     print(label)
#     print("-" * 120)
#     print("assets without photo_library_asset_unique_id:", len(missing_assets))
#     print()

#     for asset in missing_assets:
#         path = asset.get("path")
#         original_filename = asset.get("original_filename")
#         filename = asset.get("filename")
#         date = asset.get("date")
#         file_size_bytes = asset.get("file_size_bytes")
#         adjustment_signature = asset.get("adjustment_signature")

#         if original_filename is None and filename is None:
#             reason_counter["missing filename and original_filename"] += 1

#         if date is None:
#             reason_counter["missing date"] += 1

#         if path is None:
#             reason_counter["path is None"] += 1
#         elif not os.path.exists(path):
#             reason_counter["path does not exist"] += 1

#         if file_size_bytes is None:
#             reason_counter["file_size_bytes is None"] += 1

#         if adjustment_signature is None:
#             reason_counter["adjustment_signature is None"] += 1

#     print("reason counter:")
#     for reason, count in reason_counter.most_common():
#         print(f"  {reason}: {count}")

#     print()
#     print("missing asset details:")
#     print("-" * 120)

#     for index, asset in enumerate(missing_assets[:max_print], start=1):
#         path = asset.get("path")

#         print(f"{index:02d}.")
#         print("  uuid:", asset.get("uuid"))
#         print("  original_filename:", asset.get("original_filename"))
#         print("  filename:", asset.get("filename"))
#         print("  date:", asset.get("date"))
#         print("  date_added:", asset.get("date_added"))
#         print("  path:", path)
#         print("  path_exists:", None if path is None else os.path.exists(path))
#         print("  file_size_bytes:", asset.get("file_size_bytes"))
#         print("  adjustment_signature:", asset.get("adjustment_signature"))
#         print("  is_movie:", asset.get("is_movie"))
#         print("  hasadjustments:", asset.get("hasadjustments"))
#         print("  path_edited:", asset.get("path_edited"))
#         print("  asset_scope:", asset.get("asset_scope"))
#         print("  albums:", list((asset.get("albums") or {}).values()))
#         print("  folders:", list((asset.get("folders") or {}).values()))
#         print("-" * 120)

# debug_dump_assets_without_photo_library_asset_unique_id(
#     inventory_current,
#     "CURRENT DEFAULT assets without photo_library_asset_unique_id",
# )